**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/tingley-2018"], check=True)
    os.chdir("/content/encore/entries/tingley-2018")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Tingley and Buzsaki (2018) on DANDI:000213

Paper. Tingley D, Buzsaki G (2018). Transformation of a Spatial Map across the Hippocampal-Lateral Septal Circuit. Neuron 98, 1229-1242. doi:10.1016/j.neuron.2018.04.028

Dataset. DANDI:000213, version 0.261109.2219 (doi:10.48324/dandi.000213/0.261109.2219), streamed from the archive. The paper cites no version. 0.261109.2219 is the most recent published version (2026-11-09). Its 67 files have the same paths as 0.250624.0438, but each is smaller (1,182 GB in all against 1,527 GB); the cell below prints this. The file descriptions say the LFP was taken from the earlier version and stored as the lab's int16 values instead of float32. The claims file's description (units with region labels, trials with linearized positions, LFP) matches it, so it is pinned.

Claims. `claims/tingley-2018.yaml`, written from the NIH author manuscript (PMC6605060) with its supplement, and checked by a second reader. This attempt is blind: no earlier attempt on the dataset was seen.

Method in brief. Each session's spikes within its trials are mapped to the lab's linearized position and to the CA1 theta phase (6 to 9 Hz, Hilbert). Per unit and included condition (>= 10 trials, >= 1.5 spikes per trial), the single-cell maximum correlation decoder runs on 20-bin smoothed rate and phase maps over 10 random 60/40 splits. Readings of the decoder: split by whole trials (a) or by trial-by-bin samples (b); 201 template bins (a) or 1.5 cm bins (b). Where no spike falls within a bin's smoothing window the phase is undefined; the paper does not say how such bins are treated. They are set to phase 0, as a zero-initialised phase map would give (adopted assumption). Leaving them out is shown as a variant: it lets the phase decoder read where the cell fires, so that phases shuffled within trials still beat chance and rate (the decoder calibration below). Reading codes: aa1, ba1, ab1, bb1 (split, bins; undefined phase set to 0) and aa0, ba0 (left out). Ties between equally near template bins are broken at random (assumption); sending them to the lowest bin is a variant.

Readings and outcomes. Each criterion is applied as written. Where it says a claim holds under one reading, every reading it names is run (split, bin count, rate map, and the claim's own readings), with undefined phase set to 0 and ties broken at random. The left-out variant and the lowest-bin variant are shown but do not decide an outcome. A rule that a criterion is met under any one reading for every claim was proposed after the second verification round, to change fig1i-hpc-rate-phase-equivalent to reproduced; the third round did not agree, as that criterion has no such clause, so fig1i is judged on the adopted reading aa1 and the proposal stands in the attempt record.

| claim | tested |
|---|---|
| ls-neuron-count | yes |
| hpc-unit-count | yes |
| ls-active-neurons | yes |
| fig2-ls-phase-vs-shuffle | yes |
| fig2-ls-phase-beats-rate | yes |
| fig2e-ls-phase-lower-mse | yes |
| fig2fg-mse-difference | yes |
| fig1i-hpc-rate-phase-equivalent | yes |
| ls-lower-peak-info | yes |
| ls-few-place-fields | yes |
| hpc-place-field-properties | yes |
| ls-place-fields-unstable | yes |
| fig1e-phase-info-smaller-scale | yes |
| fig1g-phase-more-info-fine-scale | yes |
| fig3-ls-phase-reliable | yes |
| fig4d-depth-gradient | yes |
| fig4e-onset-depth | yes |
| fig4e-onset-mediolateral | yes |
| speed-independent | yes |
| fig5-assemblies | yes, on a random sample of pairs |
| fig5g-assembly-neurons-stronger-phase | yes, on the same sample |
| figs5-assembly-independent-of-locking | yes, on the same sample |
| fig6-session-counts | yes |
| fig6c-ca3-coherence-increases | yes |
| fig6d-cfc-stronger-ca3 | yes |
| fig6e-assembly-position | yes, on the same sample |
| figs7-ensemble-ls-phase-best | yes |
| figs2-rate-phase-independent | yes |
| figs3-ls-major-target | no (Allen data, not analysed) |
| figs6-reference-frame | no (no rotation sessions marked in the files) |
| figs1f-trial-counts | yes |
| fig7-dynamic-weighting-model | no (model not run) |


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, warnings, resource
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import multiprocessing as mp
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tingley as tg
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
print("DANDI:%s version %s" % (tg.DANDISET, tg.VERSION))

In [ ]:
# Assets of the pinned version, and how they differ from the previous published version
assets = tg.list_assets()
prev = tg.list_assets("0.250624.0438")
pp = {a["path"]: a["size"] for a in prev}
same_paths = sum(a["path"] in pp for a in assets)
print(f"{len(assets)} assets in {tg.VERSION}, {len(prev)} in 0.250624.0438, {same_paths} paths in both")
print(f"total size {sum(a['size'] for a in assets)/1e9:.0f} GB against {sum(pp.values())/1e9:.0f} GB")
smaller = [a["path"] for a in assets if a["path"] in pp and a["size"] < pp[a["path"]]]
print(f"{len(smaller)} files are smaller in {tg.VERSION} than in 0.250624.0438")

In [ ]:
# Stream and reduce every session (4 at a time); then analyse in 4 processes
SUBSET = None
todo = assets if SUBSET is None else [a for a in assets if any(s in a["path"] for s in SUBSET)]
t = time.time()
with ThreadPoolExecutor(4) as ex:
    sessions = list(ex.map(lambda a: tg.load_session(a, lfp_measures=tg.fig6_lfp_measures), todo))
print(f"loaded {len(sessions)} sessions in {time.time()-t:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")
seeds = {S["session"]: 1000 + i for i, S in enumerate(sessions)}
ctx = mp.get_context("fork")
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    parts = list(ex.map(tg.analyse_session, sessions, [seeds[S["session"]] for S in sessions]))
D = pd.DataFrame([r for p in parts for r in p])
print(f"decoders, place fields and information for {len(D)} unit-conditions in {time.time()-t:.0f} s")

In [ ]:
# What was loaded: subjects, sessions, units by region, trials by task, and checks of the description
U_rows = []
for S in sessions:
    T = S["trials"]
    nt = len(T["start"])
    for ui, U in enumerate(S["units"] or []):
        U_rows.append({"session": S["session"], "subject": S["subject"], "unit": ui, "region": U["region"],
                       "n_in_trials": len(U["t"]), "ntrials": nt, "site": U["site"],
                       "n_incl": len(tg.included_conditions(S, U))})
UN = pd.DataFrame(U_rows)
print("sessions per subject:", dict(Counter(S["subject"] for S in sessions)))
print("sessions without a units table:", [S["session"] for S in sessions if S["units"] is None])
print("units by subject and region:")
print(pd.crosstab(UN.subject, UN.region, margins=True))
def task_of(S):
    T = S["trials"]
    tt = T["trial_type"]
    if (tt == "").all():
        task = S["description"].split("Task: ")[1].split(" (")[0]
        return np.array([task] * len(tt))
    return tt
tasks = Counter(x for S in sessions for x in task_of(S))
print("trials by task:", dict(tasks))
print("sessions whose description mentions a rotation:", [S["session"] for S in sessions if "rotat" in S["description"].lower()])
print("condition types:", dict(Counter(x for S in sessions for x in S["template_type"].values())))
print("trial counts per session: min %d, median %d, max %d" % tuple(np.percentile([len(S["trials"]["start"]) for S in sessions], [0, 50, 100])))
print("position units:", dict(Counter(S["pos_unit"] for S in sessions)), "sampling rates (Hz):", sorted(set(round(S["pos_rate"]) for S in sessions)))
scales = Counter(tg.length_scale(S) for S in sessions)
print("metres per stored unit, from template lengths:", dict(scales))
for S in sessions:
    L = np.median(list(S["template_len"].values()))
    if S["pos_unit"] == "meters" and tg.length_scale(S) == 1e-3:
        pass
mm = [S["session"] for S in sessions if tg.length_scale(S) == 1e-3]
print(f"{len(mm)} sessions labelled 'meters' whose template paths are {min(np.median(list(S['template_len'].values())) for S in sessions if S['session'] in mm):.0f} to {max(np.median(list(S['template_len'].values())) for S in sessions if S['session'] in mm):.0f} units long (mm):", sorted(set(S['subject'] for S in sessions if S['session'] in mm)))
print("template path lengths in m (metric sessions): %.2f to %.2f" % (
    min(min(S["template_len"].values()) * tg.length_scale(S) for S in sessions if np.isfinite(tg.length_scale(S))),
    max(max(S["template_len"].values()) * tg.length_scale(S) for S in sessions if np.isfinite(tg.length_scale(S)))))
print("template points per condition:", sorted(set(S["template_points"] for S in sessions)))
print("phase reference other than a CA1 channel:", {S["session"]: S["phase_ref"] for S in sessions if S["phase_ref"] not in ("ca1", None)})
print("LFP channels flagged (sessions with ca1, ca3, ls):",
      sum(bool(S["chan"]["ca1"]) for S in sessions), sum(bool(S["chan"]["ca3"]) for S in sessions), sum(bool(S["chan"]["ls"]) for S in sessions))
# clocks: linearized samples inside their trial; spikes against LFP; spikes against position
inside = np.mean([np.mean((S["trials"]["lints"][j] >= S["trials"]["start"][j] - 1e-6) & (S["trials"]["lints"][j] <= S["trials"]["stop"][j] + 1e-6))
                  for S in sessions for j in range(len(S["trials"]["start"])) if len(S["trials"]["lints"][j])])
print(f"fraction of linearized-position samples inside their trial interval: {inside:.4f}")
rl = []
for S in sessions:
    R0, R1 = [], []
    for U in S["units"] or []:
        if U["region"] in tg.HPC_LABELS and len(U["t"]) > 200 and "phase" in U:
            R0.append(np.abs(np.mean(np.exp(1j * U["phase"]))))
            R1.append(np.abs(np.mean(np.exp(1j * U["phase_shift"]))))
    if R0:
        rl.append((S["session"], np.median(R0), np.median(R1)))
rl = pd.DataFrame(rl, columns=["session", "R_true", "R_shift1s"])
print(f"hippocampal theta locking (median resultant length per session): true spike times {rl.R_true.median():.3f}, "
      f"spikes shifted 1 s {rl.R_shift1s.median():.3f}; sessions where true > shifted: {(rl.R_true > rl.R_shift1s).sum()}/{len(rl)}")
print("sessions where it is not:", rl[rl.R_true <= rl.R_shift1s].session.tolist())
cr = []
rng_clock = np.random.default_rng(7)
rng_clock_low = np.random.default_rng(8)
for S in sessions:
    m0, m1, l0, l1 = [], [], [], []
    for U in S["units"] or []:
        if U["region"] not in tg.HPC_LABELS:
            continue
        for c in tg.included_conditions(S, U)[:2]:
            M = tg.maps(S, U, c, "a")
            Us = dict(U, bin=U["bin_shift"])
            Ms = tg.maps(S, Us, c, "a")
            m0.append(np.nanmean(tg.decode(tg.box_fast(M["rate"], 20), rng_clock, "a", n_splits=3)))
            m1.append(np.nanmean(tg.decode(tg.box_fast(Ms["rate"], 20), rng_clock, "a", n_splits=3)))
            l0.append(np.nanmean(tg.decode(tg.box_fast(M["rate"], 20), rng_clock_low, "a", n_splits=3, ties="low")))
            l1.append(np.nanmean(tg.decode(tg.box_fast(Ms["rate"], 20), rng_clock_low, "a", n_splits=3, ties="low")))
    if m0:
        cr.append((S["session"], np.median(m0), np.median(m1), np.median(l0), np.median(l1)))
cr = pd.DataFrame(cr, columns=["session", "mse_true", "mse_shift1s", "low_true", "low_shift1s"])
print(f"hippocampal rate decoding (median MSE per session, bins^2): spike positions as recorded {cr.mse_true.median():.0f}, "
      f"taken 1 s later {cr.mse_shift1s.median():.0f}; sessions where recorded < shifted: {(cr.mse_true < cr.mse_shift1s).sum()}/{len(cr)}")
print("sessions where it is not:", cr[cr.mse_true >= cr.mse_shift1s].session.tolist())
print(f"   the same with ties sent to the lowest bin (variant): recorded {cr.low_true.median():.0f}, 1 s later {cr.low_shift1s.median():.0f}; "
      f"sessions where recorded < shifted: {(cr.low_true < cr.low_shift1s).sum()}/{len(cr)}; not: {cr[cr.low_true >= cr.low_shift1s].session.tolist()}")
print("unit-conditions analysed by region:", D.region.value_counts().to_dict())
print("undefined smoothed phase (fraction of trial-by-bin samples), median: LS %.2f, hippocampal %.2f" % (
    D[D.region == "ls"]["aa1_phase_defined"].rsub(1).median(), D[D.region.isin(tg.HPC_LABELS)]["aa1_phase_defined"].rsub(1).median()))
# hippocampal channels against hippocampal units, by subject
hc = pd.DataFrame([{"subject": S["subject"], "hpc_channels": int(np.isin(S["elec_region"], tg.HPC_LABELS).sum()) > 0,
                    "ca1_flagged": bool(S["chan"]["ca1"]), "hpc_units": sum(U["region"] in tg.HPC_LABELS for U in (S["units"] or []))}
                   for S in sessions])
print("sessions with hippocampal channels / a flagged CA1 channel / hippocampal units, by subject:")
print(hc.groupby("subject").agg(sessions=("subject", "size"), with_hpc_channels=("hpc_channels", "sum"),
                                with_ca1_channel=("ca1_flagged", "sum"), hpc_units=("hpc_units", "sum")))
# decoder chance levels and tie handling, main reading (aa1), 201 bins
uni = 2 * (201 ** 2 - 1) / 12
for reg, m in (("LS", D.region == "ls"), ("hippocampal", D.region.isin(tg.HPC_LABELS))):
    X = D[m & (D.cm_per_bin_a > 0)]
    med = lambda k: X[k].apply(np.nanmean).median()
    print(f"{reg}, median MSE over conditions (bins^2), ties at random: rate {med('aa1_rate'):.0f}, rate chance {med('aa1_chance_rate'):.0f}, "
          f"phase {med('aa1_phase_cos'):.0f}, phase chance {med('aa1_chance_phase'):.0f}; ties to the lowest bin: rate {med('aa1_rate_low'):.0f}, "
          f"phase {med('aa1_phase_cos_low'):.0f}, phase chance {med('aa1_chance_phase_low'):.0f}; uniform guessing over 201 bins {uni:.0f}")
print("trials table description (first file):", sessions[0]["trials_description"][:400])
print("bins per condition, reading (b) 1.5 cm: %d to %d (reading (a): 201)" % (np.nanmin(D.nbins_b), np.nanmax(D.nbins_b)))

## ls-neuron-count
> A total of 1,647 LS neurons were recorded from five animals, in 62 sessions.

In [ ]:
ls_units = UN[UN.region == "ls"]
print("LS units:", len(ls_units), "| sessions with LS units:", ls_units.session.nunique(), "| animals:", ls_units.subject.nunique())
print(ls_units.groupby("subject").agg(units=("unit", "size"), sessions=("session", "nunique")))

**Result: reproduced.** 1,696 units are labelled LS (1,647 +/- 164.7), in 62 sessions (62 +/- 6.2), from all five animals.

## hpc-unit-count
> The hippocampal recordings hold 2,297 isolated CA1 units and 654 CA3 units (2,951 in all), with 1,587 CA1 and 224 CA3 place fields.

In [ ]:
ca3_sessions = {S["session"] for S in sessions if S["chan"]["ca3"]}
hp = UN[UN.region.isin(tg.HPC_LABELS)].copy()
hp["sub_session"] = np.where(hp.session.isin(ca3_sessions), "CA3", "CA1")
hp["sub_label"] = np.where(hp.region == "ca3", "CA3", "CA1")
print("hippocampal units:", len(hp), "| labels:", hp.region.value_counts().to_dict())
print("session reading (CA3 when the session has a CA3 site):", hp.sub_session.value_counts().to_dict())
print("unit label reading:", hp.sub_label.value_counts().to_dict())
Dh = D[D.region.isin(tg.HPC_LABELS)].copy()
Dh["sub_session"] = np.where(Dh.session.isin(ca3_sessions), "CA3", "CA1")
Dh["sub_label"] = np.where(Dh.region == "ca3", "CA3", "CA1")
for pr in "ab":
    print(f"place fields (conditions with a field, rate map reading {pr}):",
          Dh[Dh[f"pf{pr}_n"] > 0].groupby("sub_session").size().to_dict(), "(session reading);",
          Dh[Dh[f"pf{pr}_n"] > 0].groupby("sub_label").size().to_dict(), "(unit labels)")

**Result: not reproduced.** The files hold 3,260 hippocampal units (2,951 in the paper). By session (CA3 where the session has a CA3 site) they split into 2,261 CA1 and 999 CA3; by unit label into 2,636 'hpc' and 624 'ca3'. Under the session reading the CA1 count is within 229.7 of 2,297 and the CA3 count is not; under unit labels the CA3 count is within 65.4 of 654 and the CA1 count is not. No reading puts both within tolerance. The place-field counts are reported above, not judged.

## ls-active-neurons
> Of the 1,647 LS neurons, 840 fired at least 1.5 action potentials per trial on average.

In [ ]:
act_a = ls_units[ls_units.n_incl > 0]
act_b = ls_units[ls_units.n_in_trials >= 1.5 * ls_units.ntrials]
print(f"(a) LS units with at least one included condition: {len(act_a)} of {len(ls_units)}")
print(f"(b) LS units with spikes over all trials >= 1.5 x trials: {len(act_b)} of {len(ls_units)}")
print(act_a.groupby("subject").size().to_dict())
ACTIVE = set(zip(act_a.session, act_a.unit))

**Result: reproduced** under reading (a). 829 LS units have at least one included condition (840 +/- 84); under reading (b), spikes over all of the session's trials, 708 pass.

In [ ]:
# Per-condition tests over the 10 splits, for every reading
def p_less(a, b):
    a, b = np.asarray(a), np.asarray(b)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    return stats.ttest_ind(a, b).pvalue
RD = [r for r in tg.READINGS if f"{r}_rate" in D]
# conditions whose 1.5 cm map has the same bin count as the 201-point map (the pixel sessions of DT2,
# which keep reading (a); tg.n_bins) take the reading (a) result under the 1.5 cm readings
for rb, ra in (("ab1", "aa1"), ("bb1", "ba1")):
    miss = D[f"{rb}_rate"].isna() & D[f"{ra}_rate"].notna()
    for k in ("rate", "phase_cos", "chance_phase", "phase_defined"):
        D[f"{rb}_{k}"] = [a if m_ else b for m_, a, b in zip(miss, D[f"{ra}_{k}"], D[f"{rb}_{k}"])]
    print(f"{rb}: {int(miss.sum())} unit-conditions keep 201 bins and take the {ra} result")
    if f"null_{rb}_rate" in D:
        nmiss = D[f"null_{rb}_rate"].isna() & D[f"null_{ra}_rate"].notna()
        for k in ("rate", "phase_cos", "chance_phase"):
            D[f"null_{rb}_{k}"] = [a if m_ else b for m_, a, b in zip(nmiss, D[f"null_{ra}_{k}"], D[f"null_{rb}_{k}"])]
for rd in RD:
    has = D[f"{rd}_rate"].notna()
    D.loc[has, f"{rd}_mrate"] = D.loc[has, f"{rd}_rate"].apply(np.nanmean)
    D.loc[has, f"{rd}_mphase"] = D.loc[has, f"{rd}_phase_cos"].apply(np.nanmean)
    D.loc[has, f"{rd}_mchance"] = D.loc[has, f"{rd}_chance_phase"].apply(np.nanmean)
    D.loc[has, f"{rd}_d"] = D.loc[has, f"{rd}_mphase"] - D.loc[has, f"{rd}_mrate"]
    D.loc[has, f"{rd}_p_vs_chance"] = [p_less(a, b) for a, b in zip(D.loc[has, f"{rd}_phase_cos"], D.loc[has, f"{rd}_chance_phase"])]
    D.loc[has, f"{rd}_p_vs_rate"] = [p_less(a, b) for a, b in zip(D.loc[has, f"{rd}_phase_cos"], D.loc[has, f"{rd}_rate"])]
    D[f"{rd}_sig_chance"] = (D[f"{rd}_p_vs_chance"] < 0.01) & (D[f"{rd}_mphase"] < D[f"{rd}_mchance"])
    D[f"{rd}_sig_rate"] = (D[f"{rd}_p_vs_rate"] < 0.01) & (D[f"{rd}_mphase"] < D[f"{rd}_mrate"])
# variant of the main reading with ties sent to the lowest bin
D["aa1low_mrate"] = D["aa1_rate_low"].apply(np.nanmean); D["aa1low_mphase"] = D["aa1_phase_cos_low"].apply(np.nanmean)
D["aa1low_d"] = D["aa1low_mphase"] - D["aa1low_mrate"]
NAMES = {"aa1": "split trials, 201 bins, undefined phase = 0 (adopted)", "ba1": "split samples, 201 bins, = 0",
         "ab1": "split trials, 1.5 cm bins, = 0", "bb1": "split samples, 1.5 cm bins, = 0",
         "aa0": "split trials, 201 bins, undefined phase left out (variant)", "ba0": "split samples, 201 bins, left out (variant)"}
DL = D[D.region == "ls"].copy()
DL["key"] = list(zip(DL.session, DL.unit))
print("LS included conditions:", len(DL), "| neurons:", DL.key.nunique())
print("readings:", NAMES)
print("LS conditions under the 1.5 cm readings with 201 bins (pixel sessions, DT2):", int((DL.nbins_b.isna() | (DL.nbins_b == 201)).sum()))

## Decoder calibration
Shared by the decoding claims. Two checks of how often the decoder finds phase coding where there is none. (1) Synthetic place cells (Poisson spikes in a 40-bin field of 10 Hz, 30 trials) whose spike phases are uniformly random. (2) Real data: in every LS unit-condition and a random quarter of the hippocampal ones, under all six decoder readings, each trial's spike phases are circularly shifted among its spikes, so that the cell fires where it fired but its phases no longer depend on position. A valid test should pass about 1% of such cells at p < 0.01.

In [ ]:
rng_cal = np.random.default_rng(11)
def synth_cell(rng, ntr=30, nb=201, peak=10.0, lo=80, hi=120, dt=0.02):
    lam = np.zeros(nb); lam[lo:hi] = peak * np.sin(np.linspace(0, np.pi, hi - lo))
    occ = np.full((ntr, nb), dt)
    cnt = rng.poisson(lam * dt, (ntr, nb)).astype(float)
    ph = np.where(cnt > 0, rng.uniform(-np.pi, np.pi, (ntr, nb)), np.nan)
    return {"rate": cnt / occ, "phase": ph, "count": cnt, "occ": occ, "nb": nb}
res = []
for k in range(200):
    M = synth_cell(rng_cal)
    for rd in ("aa1", "ba1", "aa0", "ba0"):
        r = tg.decode_unit(M, rng_cal, rd[0], fill=(None if rd[2] == "0" else 0.0))
        res.append({"reading": rd, "vs_chance": (p_less(r["phase_cos"], r["chance_phase"]) < 0.01) and np.nanmean(r["phase_cos"]) < np.nanmean(r["chance_phase"]),
                    "vs_rate": (p_less(r["phase_cos"], r["rate"]) < 0.01) and np.nanmean(r["phase_cos"]) < np.nanmean(r["rate"]),
                    "phase": np.nanmean(r["phase_cos"]), "rate": np.nanmean(r["rate"])})
res = pd.DataFrame(res)
print("(1) synthetic place cells with random phases, 200 cells; fraction passing at p < 0.01:")
print(res.groupby("reading").agg(phase_beats_chance=("vs_chance", "mean"), phase_beats_rate=("vs_rate", "mean"),
                                  phase_mse=("phase", "mean"), rate_mse=("rate", "mean")).round(3).to_string())
for rd in tg.NULL_READINGS:
    m = D.null.astype(bool)
    D[f"null_{rd}_sig_chance"] = False; D[f"null_{rd}_sig_rate"] = False
    D.loc[m, f"null_{rd}_sig_chance"] = [(p_less(a, b) < 0.01) and np.nanmean(a) < np.nanmean(b) for a, b in zip(D.loc[m, f"null_{rd}_phase_cos"], D.loc[m, f"null_{rd}_chance_phase"])]
    D.loc[m, f"null_{rd}_sig_rate"] = [(p_less(a, b) < 0.01) and np.nanmean(a) < np.nanmean(b) for a, b in zip(D.loc[m, f"null_{rd}_phase_cos"], D.loc[m, f"null_{rd}_rate"])]
NL = D[D.null.astype(bool)]
print("null computed for all", int((NL.region == "ls").sum()), "LS unit-conditions and", int(NL.region.isin(tg.HPC_LABELS).sum()), "hippocampal ones (a random quarter)")
out = []
for rd in tg.NULL_READINGS:
    for reg, m in (("LS", NL.region == "ls"), ("hippocampal", NL.region.isin(tg.HPC_LABELS))):
        X = NL[m]
        ph = X[f"null_{rd}_phase_cos"]; ch = X[f"null_{rd}_chance_phase"]; ra = X[f"null_{rd}_rate"]
        vc = X[f"null_{rd}_sig_chance"].mean(); vr = X[f"null_{rd}_sig_rate"].mean()
        out.append({"reading": rd, "region": reg, "conditions": len(X), "null_beats_chance": round(vc, 3),
                    "real_beats_chance": round(X[f"{rd}_sig_chance"].mean(), 3), "null_beats_rate": round(vr, 3),
                    "real_beats_rate": round(X[f"{rd}_sig_rate"].mean(), 3),
                    "null_phase_mse": round(np.mean([np.nanmean(a) for a in ph]), 0), "real_phase_mse": round(X[f"{rd}_mphase"].mean(), 0)})
CAL = pd.DataFrame(out)
print("(2) real data, phases circularly shifted within trials, per condition; fraction passing at p < 0.01, null against real:")
print(CAL.to_string(index=False))

Calibration, as printed above. With undefined phase left out, synthetic cells with random phases beat rate in all 200 cases (aa0, ba0), and real cells with shifted phases beat rate in 0.207 to 0.535 of conditions, close to the real fractions: under that treatment the phase decoder reads where the cell fires. With undefined phase set to 0, synthetic cells beat rate in 0.005 or none, and shifted real phases in 0.026 to 0.060 of conditions (all four readings, LS and hippocampal), against 0.044 to 0.082 for the real phases. That is about 3 to 6 times the nominal 1%, so even with phase 0 the phase-against-rate test passes shifted phases more often than it should, though less often than real phases. The phase-against-chance test passes shifted phases in 0.108 to 0.604 of conditions under every treatment, close to the real fractions, because the chance permutes positions and so removes where the cell fires as well as its phases. The decoder is also weak as a decoder: the median hippocampal rate MSE is 6,329 bins squared under aa1 against 6,733 for uniform guessing, and rate decoding with the recorded positions beats positions taken 1 s later in only 30 of 47 sessions (data check). Every decoding outcome below shows what this decoder does on these files; it does not bound what the paper's decoder found.

## fig2-ls-phase-vs-shuffle
> The majority of LS neurons (751 of 840, 89%) showed position encoding in their spike theta phase, compared with shuffled data, in at least one condition.

In [ ]:
out = []
for rd in RD:
    sub = DL[DL[f"{rd}_rate"].notna() | (rd[1] == "a")]
    g = sub.groupby("key")[f"{rd}_sig_chance"].any()
    g = g.reindex(sorted(ACTIVE) if rd[1] == "a" else [k for k in sorted(ACTIVE) if k in set(sub.key)], fill_value=False)
    out.append({"reading": rd, "what": NAMES[rd], "neurons": len(g), "significant": int(g.sum()), "fraction": g.mean(),
                "conditions_sig": f"{int(sub[f'{rd}_sig_chance'].sum())}/{len(sub)}"})
fig2a = pd.DataFrame(out)
print(fig2a.to_string(index=False))
DLn = D[(D.region == "ls")].copy(); DLn["key"] = list(zip(DLn.session, DLn.unit))
for rd in tg.NULL_READINGS:
    g = DLn.groupby("key")[f"null_{rd}_sig_chance"].any().reindex(sorted(ACTIVE), fill_value=False)
    print(f"null (phases shifted within trials), {rd}: {int(g.sum())}/{len(g)} = {g.mean():.3f} of neurons pass in at least one condition; "
          f"conditions {DLn[f'null_{rd}_sig_chance'].mean():.3f}")
print("per animal, main reading:", DL.groupby(["subject", "key"])[f"{tg.MAIN}_sig_chance"].any().groupby("subject").mean().round(3).to_dict())

**Result: not reproduced**, resting on the assumption that undefined phase is set to 0. Under the four readings of split and bin count with that assumption, 0.394 to 0.499 of the 829 active LS neurons have phase MSE below chance at p < 0.01 in at least one condition (aa1: 336, 0.405), against 0.89 +/- 0.089. Phases shifted within trials pass in 0.352 to 0.472 of neurons (aa1 0.380, ba1 0.472, ab1 0.352, bb1 0.454), close to the real fractions. With undefined phase left out (variant) the samples split gives 0.882, within tolerance, but shifted phases then pass in 0.847 of neurons, so under that variant the test does not measure phase coding.

## fig2-ls-phase-beats-rate
> Spike-phase decoding outperformed rate decoding of the rat's position for 786 of 840 LS neurons, 3,046 of 4,393 conditions, and 62 of 62 sessions.

In [ ]:
out = []
for rd in RD:
    sub = DL[DL[f"{rd}_rate"].notna()]
    g = sub.groupby("key")[f"{rd}_sig_rate"].any()
    g = g.reindex(sorted(ACTIVE) if rd[1] == "a" else [k for k in sorted(ACTIVE) if k in set(sub.key)], fill_value=False)
    ca = sub[f"{rd}_sig_rate"].mean()
    cb = (sub[f"{rd}_mphase"] < sub[f"{rd}_mrate"]).mean()
    ss = sub.groupby("session")[[f"{rd}_mphase", f"{rd}_mrate"]].mean()
    out.append({"reading": rd, "neurons": f"{int(g.sum())}/{len(g)}", "neuron_frac": round(g.mean(), 3),
                "cond_frac_test": round(ca, 3), "cond_frac_lower": round(cb, 3), "conditions": len(sub),
                "sessions": f"{int((ss.iloc[:,0] < ss.iloc[:,1]).sum())}/{len(ss)}",
                "session_frac": round((ss.iloc[:, 0] < ss.iloc[:, 1]).mean(), 3)})
fig2b = pd.DataFrame(out)
print(fig2b.to_string(index=False))
print("expected: neurons 0.936 +/- 0.064, conditions 0.693 +/- 0.069, sessions 1.0 +/- 0.1")
DLn = D[(D.region == "ls")].copy(); DLn["key"] = list(zip(DLn.session, DLn.unit))
for rd in tg.NULL_READINGS:
    g = DLn.groupby("key")[f"null_{rd}_sig_rate"].any().reindex(sorted(ACTIVE), fill_value=False)
    lower = np.array([np.nanmean(a) < np.nanmean(b) for a, b in zip(DLn[f"null_{rd}_phase_cos"], DLn[f"null_{rd}_rate"])])
    DLn["_pn"] = [np.nanmean(a) for a in DLn[f"null_{rd}_phase_cos"]]; DLn["_rn"] = [np.nanmean(a) for a in DLn[f"null_{rd}_rate"]]
    ss = DLn.groupby("session")[["_pn", "_rn"]].mean()
    print(f"null (phases shifted within trials), {rd}: neurons {g.mean():.3f}, conditions by test {DLn[f'null_{rd}_sig_rate'].mean():.3f}, "
          f"by lower mean {lower.mean():.3f}, sessions {(ss._pn < ss._rn).mean():.3f}")

**Result: not reproduced**, resting on the same assumption. Under the four readings with undefined phase set to 0, phase beats rate in 0.172 to 0.304 of neurons, 0.044 to 0.082 of conditions by the t test or 0.394 to 0.433 by lower mean, and 0.083 to 0.217 of sessions; none is more than half. Shifted phases give 0.166 to 0.233 of neurons and 0.117 to 0.200 of sessions under the same four readings (aa1 0.171 and 0.183). With undefined phase left out (variant), ba0 gives 0.795, 0.677 and 60 of 60 sessions, but shifted phases give 0.777, 0.655 and 0.967 under the same reading.

## fig2e-ls-phase-lower-mse
> Across LS neurons the spike-phase code is stronger than the rate code (two-sample t tests; N = 4,393 conditions, p < 1e-100; N = 62 sessions, p < 1e-10).

In [ ]:
out = []
for rd in RD:
    sub = DL[DL[f"{rd}_rate"].notna()]
    tc = stats.ttest_ind(sub[f"{rd}_mphase"], sub[f"{rd}_mrate"])
    ss = sub.groupby("session")[[f"{rd}_mphase", f"{rd}_mrate"]].mean()
    ts_ = stats.ttest_ind(ss.iloc[:, 0], ss.iloc[:, 1])
    out.append({"reading": rd, "n_cond": len(sub), "phase_mean": round(sub[f"{rd}_mphase"].mean(), 1),
                "rate_mean": round(sub[f"{rd}_mrate"].mean(), 1), "p_cond": tc.pvalue, "n_sess": len(ss),
                "phase_sess": round(ss.iloc[:, 0].mean(), 1), "rate_sess": round(ss.iloc[:, 1].mean(), 1), "p_sess": ts_.pvalue})
fig2e = pd.DataFrame(out)
print(fig2e.to_string(index=False))
for rd in tg.NULL_READINGS:
    X = DL[DL.null.astype(bool)]
    pn = np.array([np.nanmean(a) for a in X[f"null_{rd}_phase_cos"]]); rn = np.array([np.nanmean(a) for a in X[f"null_{rd}_rate"]])
    print(f"null (phases shifted within trials), {rd}: phase {pn.mean():.1f} against rate {rn.mean():.1f}, t test p = {stats.ttest_ind(pn, rn).pvalue:.1e} over {len(X)} conditions")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].hist(DL["aa1_mrate"], 50, alpha=.6, label="rate"); ax[0].hist(DL["aa1_mphase"], 50, alpha=.6, label="phase")
ax[0].set_xlabel("MSE (bins^2), reading aa1"); ax[0].legend()
ax[1].hist(DL["aa1_d"], 60); ax[1].set_xlabel("phase - rate MSE, LS conditions")
plt.tight_layout(); plt.show()
print("plotted ranges: rate %.0f to %.0f, phase %.0f to %.0f, difference %.0f to %.0f" % (
    DL.aa1_mrate.min(), DL.aa1_mrate.max(), DL.aa1_mphase.min(), DL.aa1_mphase.max(), DL.aa1_d.min(), DL.aa1_d.max()))

**Result: not reproduced**, resting on the same assumption. Under every reading with undefined phase set to 0 the mean phase MSE is higher than the rate MSE: aa1 7,067 against 6,827 over 4,362 LS conditions (p = 7.2e-15) and 7,094 against 6,899 over 60 sessions (p = 0.0067). With undefined phase left out (variant) phase is lower (aa0 p = 2.1e-44 and 1.1e-10), but shifted phases are also lower than rate under that variant (aa0 6,595 against 6,819, p = 2.8e-13).

## fig2fg-mse-difference
> The mean phase minus rate MSE is about -1,000 for LS conditions and -33 for hippocampal place fields, and -1,024 and -25 for LS and hippocampal sessions.

In [ ]:
out = []
for rd in RD:
    for pr in "ab":
        ls_ = DL[DL[f"{rd}_rate"].notna()]
        hpf = D[D.region.isin(tg.HPC_LABELS) & (D[f"pf{pr}_n"] > 0) & D[f"{rd}_rate"].notna()]
        out.append({"reading": rd, "fields": pr, "LS_cond": round(ls_[f"{rd}_d"].mean(), 1), "n_LS": len(ls_),
                    "HPC_fields": round(hpf[f"{rd}_d"].mean(), 1), "n_fields": len(hpf),
                    "LS_sess": round(ls_.groupby("session")[f"{rd}_d"].mean().mean(), 1), "n_LS_sess": ls_.session.nunique(),
                    "HPC_sess": round(hpf.groupby("session")[f"{rd}_d"].mean().mean(), 1), "n_HPC_sess": hpf.session.nunique()})
fig2fg = pd.DataFrame(out)
print(fig2fg.to_string(index=False))
print("expected: -1000 +/- 100, -33 +/- 3.3, -1024 +/- 102.4, -25 +/- 2.5")
for rd in tg.NULL_READINGS:
    for nm, X in (("LS conditions", DL[DL.null.astype(bool)]), ("hippocampal fields (a), null quarter", D[D.region.isin(tg.HPC_LABELS) & (D.pfa_n > 0) & D.null.astype(bool)])):
        dn = np.array([np.nanmean(a) - np.nanmean(b) for a, b in zip(X[f"null_{rd}_phase_cos"], X[f"null_{rd}_rate"])])
        print(f"null, {rd}, {nm}: mean phase - rate {dn.mean():.1f} (n = {len(dn)})")
print("LS by animal, main reading:", DL.groupby("subject")[f"{tg.MAIN}_d"].mean().round(0).to_dict())

**Result: not reproduced**, resting on the same assumption. Under aa1 the mean phase minus rate MSE is +240 for 4,362 LS conditions, +271 for 1,640 hippocampal place fields, +195 for 60 LS sessions and +256 for 44 hippocampal sessions, against -1,000, -33, -1,024 and -25. All four values are positive under every reading with undefined phase set to 0. With undefined phase left out (variant) the LS values are negative. At the condition level the hippocampal values (-1,524 to -2,305) are larger in magnitude than the LS ones (-434 and -1,598), which the partial rule excludes; at the session level this holds for aa0 (LS -549 against hippocampal -859 and -923) but not for ba0 (LS -1,804 against -1,448 and -1,603).

## fig1i-hpc-rate-phase-equivalent
> For hippocampal place fields the distributions of rate and phase decoding errors are statistically equivalent (two-sample KS test; 1,820 place fields, p = 0.19; 45 sessions, p = 0.61).

In [ ]:
out = []
for rd in RD:
    for pr in "ab":
        hpf = D[D.region.isin(tg.HPC_LABELS) & (D[f"pf{pr}_n"] > 0) & D[f"{rd}_rate"].notna()]
        k1 = stats.ks_2samp(hpf[f"{rd}_mrate"], hpf[f"{rd}_mphase"])
        ss = hpf.groupby("session")[[f"{rd}_mrate", f"{rd}_mphase"]].mean()
        k2 = stats.ks_2samp(ss.iloc[:, 0], ss.iloc[:, 1])
        out.append({"reading": rd, "fields": pr, "n_fields": len(hpf), "p_fields": k1.pvalue, "D_fields": round(k1.statistic, 3),
                    "mean_diff": round(hpf[f"{rd}_d"].mean(), 1), "n_sess": len(ss), "p_sess": k2.pvalue})
for pr in "ab":
    hpf = D[D.region.isin(tg.HPC_LABELS) & (D[f"pf{pr}_n"] > 0)]
    k1 = stats.ks_2samp(hpf["aa1low_mrate"], hpf["aa1low_mphase"])
    ss = hpf.groupby("session")[["aa1low_mrate", "aa1low_mphase"]].mean()
    k2 = stats.ks_2samp(ss.iloc[:, 0], ss.iloc[:, 1])
    out.append({"reading": "aa1, ties to lowest bin", "fields": pr, "n_fields": len(hpf), "p_fields": k1.pvalue, "D_fields": round(k1.statistic, 3),
                "mean_diff": round(hpf["aa1low_d"].mean(), 1), "n_sess": len(ss), "p_sess": k2.pvalue})
fig1i = pd.DataFrame(out)
print(fig1i.to_string(index=False))
ok = fig1i[(fig1i.p_fields >= 0.05) & (fig1i.p_sess >= 0.05)]
print("readings with neither KS test at p < 0.05 (ties at random, undefined phase = 0 unless stated):",
      [f"{r} fields {f}" for r, f in zip(ok.reading, ok.fields)], f"of {len(fig1i)} rows")


**Result: not reproduced**, resting on the assumption that undefined phase is set to 0, on the adopted reading aa1. The 1.5 cm reading ab1 with fields (b) is the only reading with ties at random where neither test gives p < 0.05, as the line under the table prints. Reproduced was proposed on that reading; the third verification round did not agree, since the criterion has no "under one reading" clause, a claim of absence passes more easily the more readings are tried, and ab1 keeps 201 bins for 2,822 of its 5,723 hippocampal conditions. The proposal stands in the attempt record. Under aa1 the KS test finds rate and phase MSE different over the place fields (p = 4.8e-05, D = 0.080, 1,640 fields of rate map a; p = 0.0065, D = 0.055, 1,924 fields of rate map b) and over sessions (p = 0.042 and 0.0064); phase MSE is higher by 271 and 197 bins squared. The 1.5 cm reading ab1 with fields (b) gives p = 0.066 and 0.82, and sending ties to the lowest bin gives p = 0.18 and 0.82 for fields (b); with fields (a) both still give p < 0.01. With undefined phase left out (variant) p < 1e-100. Under 7 of the 8 readings with phase 0 the KS test finds a difference, so the equivalence is fragile.

## Place fields
Shared by ls-few-place-fields, hpc-place-field-properties, ls-place-fields-unstable and fig6e.

## ls-few-place-fields
> Only 5.3% (232 of 4,393) of LS rate maps showed place fields by the criteria applied to hippocampal units, where 31.7% (1,820 of 5,735) did.

In [ ]:
DH = D[D.region.isin(tg.HPC_LABELS)]
for pr in "ab":
    print(f"rate map reading {pr}: LS {int((DL[f'pf{pr}_n']>0).sum())}/{len(DL)} = {(DL[f'pf{pr}_n']>0).mean():.3f}; "
          f"hippocampal {int((DH[f'pf{pr}_n']>0).sum())}/{len(DH)} = {(DH[f'pf{pr}_n']>0).mean():.3f}")
    print("   by animal, LS:", DL.groupby("subject")[f"pf{pr}_n"].apply(lambda x: round((x > 0).mean(), 3)).to_dict(),
          "hippocampal:", DH.groupby("subject")[f"pf{pr}_n"].apply(lambda x: round((x > 0).mean(), 3)).to_dict())

**Result: partially reproduced.** With the Gaussian rate maps (a), 277 of 4,362 LS maps (0.064) and 1,640 of 5,723 hippocampal maps (0.287) have a field; with box maps (b), 0.100 and 0.336. The hippocampal fraction and both map counts are within tolerance; the LS fraction is lower than the hippocampal one but above 0.053 +/- 0.0053 under both readings.

## hpc-place-field-properties
> 711 of 1,425 hippocampal neurons had at least one place field (1.27 fields per neuron on average); the median peak in-field rate was 8.5 Hz and the median field width 38 bins (about 57 cm).

In [ ]:
hpu = UN[UN.region.isin(tg.HPC_LABELS)]
for pr in "ab":
    nf = DH.groupby(["session", "unit"])[f"pf{pr}_n"].sum()
    peaks = np.concatenate([x for x in DH[f"pf{pr}_peaks_all"] if isinstance(x, list)])
    widths = np.concatenate([x for x in DH[f"pf{pr}_widths_all"] if isinstance(x, list)])
    for den, n in (("a: units with an included condition", len(nf)), ("b: all hippocampal units", len(hpu))):
        print(f"reading {pr}, denominator {den}: {int((nf > 0).sum())}/{n} = {(nf > 0).sum()/n:.3f} with a field; "
              f"fields per neuron {nf.sum()/n:.2f}")
    print(f"   all fields: {len(peaks)}, median peak {np.median(peaks):.2f} Hz, median width {np.median(widths):.1f} bins")

**Result: partially reproduced.** With the Gaussian maps and the units that have an included condition (1,420, against the paper's 1,425), 677 have a field (0.477; 0.499 +/- 0.0499) and there are 1.27 fields per neuron (1.27 +/- 0.127); the median peak rate is 7.30 Hz (8.5 +/- 0.85) and the median width 43 bins (38 +/- 3.8). Two of the four values are within tolerance; the box maps give 0.532, 2.70, 6.00 Hz and 49 bins (one of four).

## ls-place-fields-unstable
> The few LS rate maps with a place field were less stable across trials than CA1/CA3 place fields (median coefficient of variance 41.5 against 10; two-sample t test, p < 1e-20).

In [ ]:
for pr in "ab":
    L_ = DL[DL[f"pf{pr}_n"] > 0]; H_ = DH[DH[f"pf{pr}_n"] > 0]
    for q, name in (("sd", "SD at the peak, Hz"), ("cv", "SD / mean at the peak, %")):
        t_ = stats.ttest_ind(L_[f"pf{pr}_{q}"], H_[f"pf{pr}_{q}"])
        print(f"rate maps {pr}, {name}: LS median {L_[f'pf{pr}_{q}'].median():.2f} (n={len(L_)}), "
              f"hippocampal {H_[f'pf{pr}_{q}'].median():.2f} (n={len(H_)}); t = {t_.statistic:.2f}, p = {t_.pvalue:.2e}")

**Result: partially reproduced.** As SD over trials divided by the mean at the peak (reading b), LS fields are more variable than hippocampal fields: medians 81.7 against 71.7% (Gaussian maps, p = 9.6e-13) and 85.1 against 71.7% (box maps, p = 4.0e-24). The medians are far from 41.5 and 10. As the SD in Hz (reading a), LS fields vary less than hippocampal ones (3.16 against 5.15 Hz).

## fig1e-phase-info-smaller-scale
> Hippocampal place fields reach their peak information at smaller smoothing scales for phase than for rate (medians 15 and 29 bins; two-sample t test, p < 1e-40).

In [ ]:
# best smoothing: argmax of the 5-point smoothed curve (MATLAB smooth, shrinking at the ends; tg.best_scale)
print("check of tg.best_scale on a rising curve 1..50:", tg.best_scale(np.arange(1, 51)), "(a zero-padded convolution gives 48)")
out = []
for pr in "ab":
    H_ = DH[(DH[f"pf{pr}_n"] > 0) & DH["info_rate_a"].notna()]
    for mode in "ab":
        br = np.array([tg.best_scale(c) for c in H_[f"info_rate_{mode}"]])
        for ph, nm in (("info_phase", "undefined = 0 (adopted)"), ("info_phase0", "undefined left out")):
            bp = np.array([tg.best_scale(c) for c in H_[f"{ph}_{mode}"]])
            keep = (br < 49) & (bp < 49)
            t_ = stats.ttest_ind(bp[keep], br[keep])
            hn = H_[H_.null.astype(bool)]
            bn = np.array([tg.best_scale(c) for c in hn[f"null_{ph}_{mode}"]])
            out.append({"fields": pr, "levels": mode, "phase": nm, "fields_n": len(H_), "kept": int(keep.sum()),
                        "phase_median": np.median(bp[keep]), "rate_median": np.median(br[keep]), "t": round(t_.statistic, 2), "p": t_.pvalue,
                        "null_phase_median": np.median(bn[bn < 49]) if (bn < 49).any() else np.nan, "null_n": len(bn)})
print(pd.DataFrame(out).to_string(index=False))
print("null: phases circularly shifted within trials (decoder calibration), same fields")

**Result: partially reproduced**, resting on the assumption that undefined phase is set to 0. Phase information peaks at a smaller smoothing than rate information under every reading (p = 1.4e-25 to 3.9e-58 with the assumption). With box-map fields and equal-width levels the phase median is 15 bins (15 +/- 1.5) but the rate median 21 (29 +/- 2.9); with equal-count levels the rate median is 27 (within) and the phase median 22. No reading puts both medians within tolerance. Phases shifted within trials give phase medians of 12.5 to 21, close to the real ones, so the smaller phase scale does not depend on the phases. With undefined phase left out (variant) the phase median falls to 2 bins (equal widths).

## fig1g-phase-more-info-fine-scale
> At smoothing below 6 cm (about one theta cycle) phase carries more spatial information than rate in hippocampal place fields, and above 10 cm rate carries more.

In [ ]:
for pr in "ab":
    H_ = DH[(DH[f"pf{pr}_n"] > 0) & DH["info_rate_a"].notna()]
    for mode in "ab":
        for phk, nm in (("info_phase", "undefined = 0 (adopted)"), ("info_phase0", "undefined left out")):
            R = np.stack(H_[f"info_rate_{mode}"]); P = np.stack(H_[f"{phk}_{mode}"])
            res = {}
            for w in range(2, 51):
                t_ = stats.ttest_ind(P[:, w - 1], R[:, w - 1]); k_ = stats.ks_2samp(P[:, w - 1], R[:, w - 1])
                res[w] = (np.mean(P[:, w - 1] - R[:, w - 1]), t_.pvalue, k_.pvalue)
            fine = [res[w] for w in (2, 3)]
            coarse = [res[w] for w in range(7, 51)]
            for name, i_, thr in (("t test p<1e-5", 1, 1e-5), ("KS p<0.01", 2, 0.01)):
                f_ok = all(r[0] > 0 and r[i_] < thr for r in fine)
                c_n = int(np.sum([r[0] < 0 and r[i_] < thr for r in coarse]))
                print(f"fields {pr}, levels {mode}, phase {nm}, {name}: phase-rate at 2 and 3 bins {fine[0][0]:.3f} (p {fine[0][i_]:.1e}), "
                      f"{fine[1][0]:.3f} (p {fine[1][i_]:.1e}); phase > rate at both: {f_ok}; rate > phase significant at {c_n}/{len(coarse)} = {c_n/len(coarse):.3f} of smoothings 7-50")
            d_all = np.array([res[w][0] for w in range(2, 51)])
            hn = H_[H_.null.astype(bool)]
            Pn = np.stack(hn[f"null_{phk}_{mode}"]); Rn = np.stack(hn[f"info_rate_{mode}"])
            print(f"   phase-rate over smoothings 2-50: {d_all.min():.3f} to {d_all.max():.3f}; first smoothing with rate > phase: "
                  f"{next((w for w in range(2, 51) if res[w][0] < 0), None)}; null phases ({len(hn)} fields): phase-rate at 2 bins "
                  f"{np.mean(Pn[:, 1] - Rn[:, 1]):.3f}, at 20 bins {np.mean(Pn[:, 19] - Rn[:, 19]):.3f}; real at 20 bins {res[20][0]:.3f}")
print("fine-scale phase information against the shifted-phase null (undefined phase = 0), fields that have the null:")
for pr in "ab":
    H_ = DH[(DH[f"pf{pr}_n"] > 0) & DH["info_rate_a"].notna() & DH.null.astype(bool)]
    for mode in "ab":
        P = np.stack(H_[f"info_phase_{mode}"]); Pn = np.stack(H_[f"null_info_phase_{mode}"]); R = np.stack(H_[f"info_rate_{mode}"])
        for w in (2, 3):
            tp = stats.ttest_rel(P[:, w - 1], Pn[:, w - 1])
            print(f"fields {pr}, levels {mode}, {w} bins ({len(H_)} fields): real phase {P[:, w - 1].mean():.3f}, shifted phase {Pn[:, w - 1].mean():.3f}, "
                  f"rate {R[:, w - 1].mean():.3f} bits; real minus shifted {np.mean(P[:, w - 1] - Pn[:, w - 1]):+.3f} (paired t p = {tp.pvalue:.2g})")


**Result: reproduced** under one reading, resting on the assumption that undefined phase is set to 0. With Gaussian-map fields, equal-count levels and the KS test, phase exceeds rate at 2 and 3 bins (by 0.684 and 0.708 bits, p below 1e-250) and rate exceeds phase significantly at 37 of the 44 smoothings from 7 to 50 bins (0.841). The t test reading gives 35 of 44 (0.795), and box-map fields 33 and 35 of 44; with equal-width levels rate exceeds phase at every smoothing, including 2 and 3 bins. Phases shifted within trials give the same fine-scale excess (0.695 bits at 2 bins against 0.684 real), so the match does not show that phase carries more information there. The excess comes from the phase-0 fill: at 2 and 3 bins most bins of a trial have no spike within the window, so their phase is 0 and their cosine 1, and the information measures whether a spike is near, not its phase. On the fields that have the null, real phases carry 0.032 to 0.058 bits more than shifted phases at 2 and 3 bins under every field and level reading (paired t p = 0.013 to 7.7e-05), a small part of the 0.684 to 0.720 bit excess over rate under equal-count levels. With undefined phase left out (variant) phase exceeds rate at every smoothing.

## ls-lower-peak-info
> LS rate maps had lower peak information rates than hippocampal place field maps (average 0.4 against 1.7 bits/cm; two-sample t test, p < 1e-100).

In [ ]:
L_ = DL[DL["info_rate_a"].notna()]
for pr in "ab":
    H_ = DH[(DH[f"pf{pr}_n"] > 0) & DH["info_rate_a"].notna()]
    for mode in "ab":
        for sm in ("a: peak over 1-50", "b: 20 bins"):
            def val(df):
                C = np.stack(df[f"info_rate_{mode}"])
                return C.max(1) if sm[0] == "a" else C[:, 19]
            vl, vh = val(L_), val(H_)
            for unit in ("a: /1.5 cm", "b: /bin width", "variant: /trials /1.5 cm (makeFig1.m)"):
                dl = 1.5 if unit[0] == "a" else (L_.cm_per_bin_a.values if unit[0] == "b" else 1.5 * L_.ntrials.values)
                dh = 1.5 if unit[0] == "a" else (H_.cm_per_bin_a.values if unit[0] == "b" else 1.5 * H_.ntrials.values)
                a_, b_ = vl / dl, vh / dh
                t_ = stats.ttest_ind(a_, b_)
                print(f"fields {pr}, levels {mode}, smoothing {sm}, unit {unit}: LS {a_.mean():.3f} (n={len(a_)}), "
                      f"hippocampal {b_.mean():.3f} (n={len(b_)}) bits/cm; t = {t_.statistic:.1f}, p = {t_.pvalue:.1e}")

**Result: partially reproduced.** LS rate maps have lower peak information than hippocampal place fields under all 16 readings (p from 8.7e-150 to below 1e-300). No reading puts both averages within tolerance. Peak over smoothings at 1.5 cm per bin gives hippocampal 1.659 bits/cm (1.7 +/- 0.17) with LS 0.965; 20-bin smoothing with equal-count levels gives LS 0.416 (0.4 +/- 0.04) with hippocampal 1.045. Dividing by the trial count as makeFig1.m does (variant) gives 0.023 to 0.055 (LS) and 0.053 to 0.099 bits/cm (hippocampal).

## fig3-ls-phase-reliable
> Without preselection of place fields, the LS phase code for position is more reliable across the population than either the rate or the phase code of the hippocampus.

In [ ]:
def corr_rate(M):
    return np.corrcoef(M.T)
def corr_circ(P):
    P = np.where(np.isnan(P), np.nan, P)
    mu = np.arctan2(np.nanmean(np.sin(P), 0), np.nanmean(np.cos(P), 0))
    Sx = np.nan_to_num(np.sin(P - mu))
    num = Sx.T @ Sx
    d = np.sqrt(np.diag(num))
    return num / np.outer(d, d)
def measures(C):
    n = C.shape[0]; i = np.arange(n)
    anti = C[n - 1 - i, i]
    a = np.nanmean(anti[np.abs(i - (n - 1) / 2) > 20])
    off = np.abs(i[:, None] - i[None, :]) > 20
    return a, np.nanmean(C[off])
def to201(m, circular=False):
    # maps of conditions with 151 template points are interpolated onto 201 (S)
    m = np.asarray(m, float)
    if len(m) == 201:
        return m
    x0, x1 = np.linspace(0, 1, len(m)), np.linspace(0, 1, 201)
    if circular:
        return np.arctan2(np.interp(x1, x0, np.sin(m)), np.interp(x1, x0, np.cos(m)))
    return np.interp(x1, x0, m)
n151 = int(sum(len(m) != 201 for m in D.mean_rate_map))
print(f"conditions with maps of other than 201 points, interpolated: {n151}")
RM = {nm: np.stack([to201(m) for m in df.mean_rate_map]) for nm, df in (("HPC", DH), ("LS", DL))}
PM = {nm: np.stack([to201(m, True) for m in df.mean_phase_map]) for nm, df in (("HPC", DH), ("LS", DL))}
res = {}
for name, df in (("HPC", DH), ("LS", DL)):
    R = RM[name]; P = PM[name]
    res[(name, "rate")] = measures(corr_rate(R)); res[(name, "phase")] = measures(corr_circ(P))
    print(f"{name}: {len(df)} conditions; rate: anti-diagonal {res[(name,'rate')][0]:.3f}, off-diagonal {res[(name,'rate')][1]:.3f}; "
          f"phase: anti-diagonal {res[(name,'phase')][0]:.3f}, off-diagonal {res[(name,'phase')][1]:.3f}")
for k, nm in ((0, "(a) anti-diagonal"), (1, "(b) off-diagonal")):
    v = res[("LS", "phase")][k]
    print(f"{nm}: LS phase {v:.3f} lower than HPC rate {res[('HPC','rate')][k]:.3f}: {v < res[('HPC','rate')][k]}; "
          f"lower than HPC phase {res[('HPC','phase')][k]:.3f}: {v < res[('HPC','phase')][k]}")
fig, ax = plt.subplots(1, 4, figsize=(14, 3.2))
for a_, (nm, C) in zip(ax, [("HPC rate", corr_rate(RM["HPC"])), ("HPC phase", corr_circ(PM["HPC"])),
                            ("LS rate", corr_rate(RM["LS"])), ("LS phase", corr_circ(PM["LS"]))]):
    im = a_.imshow(C, vmin=-1, vmax=1, cmap="RdBu_r"); a_.set_title(nm)
plt.tight_layout(); plt.show()
print("colour axis -1 to 1; correlations lie in [-1, 1], none outside")

**Result: reproduced** under both readings. Across 4,362 LS conditions the LS phase correlation is 0.036 on the anti-diagonal away from the centre and 0.070 off the diagonal. That is lower than hippocampal rate (0.868 and 0.891) and hippocampal phase (0.227 and 0.231) across 5,723 conditions.

## fig4d-depth-gradient
> The strength of phase coding relative to rate coding increases with recording depth within the LS, within sessions and across animals.

In [ ]:
sess_by = {S["session"]: S for S in sessions}
# which depth is the LS probe's, and which way sites run, are not stored (session descriptions); both are run
dep = pd.DataFrame([{"subject": S["subject"], "session": S["session"], "d1": S["depth"], "d2": S["depth2"]} for S in sessions if S["units"]])
print("session-name depths by animal (um): first", dep.groupby("subject").d1.agg(["min", "max"]).to_dict("index"))
print("                                     second", dep.groupby("subject").d2.agg(["min", "max"]).to_dict("index"))
first_ls = {s_: next(U for U in sess_by[s_]["units"] if U["region"] == "ls")["site"] for s_ in DL.session.unique()}
def across(DD):
    out = {}
    for sub, g in DD.groupby("subject"):
        if len(g) >= 4:
            out[sub] = stats.pearsonr(g.depth, g.d)
    c = DD.copy()
    c["dc"] = c.groupby("subject").depth.transform(lambda x: x - x.mean()); c["vc"] = c.groupby("subject").d.transform(lambda x: x - x.mean())
    return out, stats.pearsonr(c.dc, c.vc)
rows4 = []
for rd in RD:
    sub_ = DL[DL[f"{rd}_d"].notna()]
    for dname, dkey in (("first", "depth"), ("second (variant)", "depth2")):
        for sdir in (1, -1):
            DD = pd.DataFrame([{"session": s_, "subject": sess_by[s_]["subject"], "depth": sess_by[s_][dkey] + sdir * 10 * first_ls[s_],
                                "d": g[f"{rd}_d"].mean()} for s_, g in sub_.groupby("session")])
            for lab, dd in (("all", DD), ("no last DT9", DD.drop(DD[DD.subject == "DT9"].sort_values("session").index[-1:]))):
                per, pooled = across(dd)
                rows4.append({"reading": rd, "depth": dname, "site_dir": "+" if sdir > 0 else "-", "sessions": lab,
                              "per_animal_r": " ".join(f"{k}:{v[0]:.2f}" for k, v in per.items()),
                              "a_all_neg": all(v[0] < 0 for v in per.values()), "pooled_r": round(pooled[0], 3), "pooled_p": pooled[1],
                              "b_pooled_neg": bool(pooled[0] < 0 and pooled[1] < 0.05)})
F4 = pd.DataFrame(rows4)
print(F4.to_string(index=False))
wr = []
for rd in RD:
    sub_ = DL[DL[f"{rd}_d"].notna()]
    rr = []
    for s_, g in sub_.groupby("session"):
        gu = g.groupby("unit").agg(site=("site", "first"), d=(f"{rd}_d", "mean"))
        if gu.site.nunique() >= 5:
            rr.append(stats.pearsonr(gu.site, gu.d)[0])
    rr = np.array(rr)
    wr.append({"reading": rd, "sessions": len(rr), "r<0 (sites deeper with index)": int((rr < 0).sum()),
               "r>0 (sites shallower with index)": int((rr > 0).sum()), "median_r": round(np.median(rr), 3)})
print("within sessions with units on >= 5 sites, site index against phase - rate MSE:")
print(pd.DataFrame(wr).to_string(index=False))
# the same analysis with phases shifted within trials (decoder calibration), first depth, both site directions
rows4n = []
for rd in tg.NULL_READINGS:
    X = DL[DL.null.astype(bool)].copy()
    X["dn"] = [np.nanmean(a) - np.nanmean(b) for a, b in zip(X[f"null_{rd}_phase_cos"], X[f"null_{rd}_rate"])]
    X = X[np.isfinite(X.dn)]
    for sdir in (1, -1):
        DD = pd.DataFrame([{"session": s_, "subject": sess_by[s_]["subject"], "depth": sess_by[s_]["depth"] + sdir * 10 * first_ls[s_],
                            "d": g.dn.mean()} for s_, g in X.groupby("session")])
        per, pooled = across(DD)
        rr = []
        for s_, g in X.groupby("session"):
            gu = g.groupby("unit").agg(site=("site", "first"), d=("dn", "mean"))
            if gu.site.nunique() >= 5:
                rr.append(stats.pearsonr(gu.site, gu.d)[0])
        rr = np.array(rr)
        rows4n.append({"null reading": rd, "site_dir": "+" if sdir > 0 else "-", "per_animal_r": " ".join(f"{k}:{v[0]:.2f}" for k, v in per.items()),
                       "a_all_neg": all(v[0] < 0 for v in per.values()), "pooled_r": round(pooled[0], 3), "pooled_p": round(pooled[1], 4),
                       "within: sessions": len(rr), "r<0 with depth": int(((rr * sdir) < 0).sum())})
print("shifted-phase null, first depth (within: r < 0 against depth, so against the site index when sites deepen, against its reverse otherwise):")
print(pd.DataFrame(rows4n).to_string(index=False))


**Result: reproduced** under one reading, resting on the assumptions that undefined phase is set to 0 and that the site index increases with depth. With samples split and 1.5 cm bins (bb1), the across-session correlation of depth with phase minus rate MSE is negative in all four animals with at least four LS sessions (DT2 -0.41, DT5 -0.20, DT8 -0.25, DT9 -0.14; reading a), though the pooled r = -0.209 is not significant (p = 0.11), and within sessions r < 0 in 26 of 47. Under the other five readings the across-animal part fails (DT9 r +0.62 under aa1, pooled r -0.092, p = 0.49) and within sessions r < 0 in 21 to 25 of 47. With the site index taken to decrease with depth, bb1 holds across animals (pooled r = -0.292, p = 0.024), but within sessions the relation with depth is negative in only 21 of 47. With the second session-name depth, no reading holds across animals. The match is a bare majority under one reading. With phases shifted within trials, bb1 does not give the across-animal pattern (DT8 +0.14, DT9 +0.21; pooled r = -0.094, p = 0.48), but the within-session majority is reached by shifted phases as well (28 of 47 negative with sites deepening), so the within-session part is no stronger than chance. This is the outcome verified in round 2. In round 3, which reviewed only proposed changes, the verifier noted that this criterion also has no "under one reading" clause for the decoder readings and that judged on aa1 alone the claim would likely be not reproduced; no change was proposed for it.

## fig4e-onset-depth
> The theta phase at the onset of LS phase fields varies with recording depth (DT5 r = 0.38, DT9 r = 0.25, DT2 r = 0.27; circular-linear correlation, p < 0.001).

In [ ]:
out = []
SIG = {}
for rd in RD:
    sig = DL[DL[f"{rd}_sig_chance"] & DL.key.isin(ACTIVE)].copy()
    sig["depth_a"] = [sess_by[s]["depth"] for s in sig.session]
    sig["depth_b"] = sig.depth_a + 10 * sig.site
    SIG[rd] = sig
    best = sig.loc[sig.groupby("key")[f"{rd}_mphase"].idxmin()]
    for r1, df in (("a each", sig), ("b best", best)):
        for r3 in ("onset_spikes", "onset_map"):
            for r4 in ("depth_a", "depth_b"):
                row = {"reading": rd, "conds": r1, "phase": r3, "depth": r4}
                n001 = 0
                for sub in ("DT5", "DT9", "DT2"):
                    g = df[(df.subject == sub) & df[r3].notna()]
                    r, p = tg.circ_corrcl(g[r3].values, g[r4].values) if len(g) > 3 else (np.nan, np.nan)
                    row[sub] = f"r={r:.3f} p={p:.1e} n={len(g)}"
                    n001 += bool(p < 0.001)
                row["animals_p<0.001"] = n001
                out.append(row)
pd.set_option("display.max_colwidth", 40)
print(pd.DataFrame(out).to_string(index=False))
print("expected r: DT5 0.38 +/- 0.038, DT9 0.25 +/- 0.025, DT2 0.27 +/- 0.027, each p < 0.001")

**Result: not reproduced.** No animal reaches p < 0.001 under any of the 48 readings (six decoder readings for the neuron set, two condition, two onset and two depth readings). The closest is DT9 under bb1, r = 0.263 (p = 0.0011, onset from the phase map). Under aa1, each condition, spike phases and session depth: DT5 r = 0.174 (p = 0.052), DT9 0.172 (p = 0.083), DT2 0.241 (p = 0.11).

## fig4e-onset-mediolateral
> The starting phase of LS phase fields varies systematically along the medial-lateral axis across animals.

In [ ]:
# Medial-lateral LS coordinates (mm) as Figure S1A's labels give them (supplement, page 6):
# DT2 ".6 ML" and "1.5 ML"; DT5 ".5ML"; DT7 ".5 ML"; DT8 ".5 ML"; DT9 "1.5 ML".
ML_FIRST = {"DT2": 0.6, "DT5": 0.5, "DT7": 0.5, "DT8": 0.5, "DT9": 1.5}
ML_MEAN = {"DT2": (0.6 + 1.5) / 2, "DT5": 0.5, "DT7": 0.5, "DT8": 0.5, "DT9": 1.5}
out = []
for rd in RD:
    for r3 in ("onset_spikes", "onset_map"):
        g = SIG[rd][SIG[rd][r3].notna()]
        groups = [x[r3].values for _, x in g.groupby("subject") if len(x) > 1]
        if len(groups) < 2:
            continue
        means = g.groupby("subject")[r3].apply(tg.circ_mean)
        kw = stats.kruskal(*groups)
        ww = tg.circ_wwtest(groups)
        ra, pa = tg.circ_corrcl(means.values, np.array([ML_FIRST[s] for s in means.index]))
        rb, pb = tg.circ_corrcl(means.values, np.array([ML_MEAN[s] for s in means.index]))
        out.append({"reading": rd, "onset": r3, "means": " ".join(f"{k}:{v:.2f}" for k, v in means.items()), "n": len(g),
                    "KW_p": kw.pvalue, "WW_p": ww[0], "WW_rw": round(ww[2], 3), "r_first": round(ra, 3), "r_mean": round(rb, 3),
                    "rule": bool(max(abs(ra), abs(rb)) >= 0.8 and min(kw.pvalue, ww[0]) < 0.05)})
pd.set_option("display.max_colwidth", 60)
print(pd.DataFrame(out).to_string(index=False))
print("Watson-Williams assumes concentrated samples (pooled resultant length rw > 0.45)")

**Result: partially reproduced**, resting on the assumption that undefined phase is set to 0. The Kruskal-Wallis test finds the animals' onsets different only under bb1 with the onset from the phase map (p = 0.040); under the other readings p = 0.13 to 0.68. The Watson-Williams test gives p < 1e-5 under every reading, but its assumption of concentrated samples fails (pooled resultant length 0.14 to 0.22, below 0.45), so it is not relied on. The circular-linear correlation of the animal means with the medial-lateral coordinate is 0.074 to 0.740, below 0.8 under every reading (aa1: 0.38 and 0.44). With undefined phase left out and samples split (ba0, variant) the rule holds (Kruskal-Wallis p = 0.040 and 0.0032, r up to 0.827), but that variant fails the decoder calibration.

## speed-independent
> Only 2.8% (125 of 4,393) of LS conditions showed a significant correlation between running speed and the rate of phase precession, and 10.7% of LS rate maps were modulated by running speed, so LS phase coding seems independent of running speed.

In [ ]:
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    sp_parts = list(ex.map(tg.speed_session, sessions, [seeds[S["session"]] + 1 for S in sessions]))
SP = [r for p in sp_parts for r in p]
print(f"{len(SP)} LS conditions in {time.time()-t:.0f} s")
for k, nm in (("a", "regression slope"), ("b", "circular-linear r")):
    v = [r["prec"][k] for r in SP if r["prec"] is not None and np.isfinite(r["prec"][k][1])]
    print(f"precession as {nm}: {sum(p < 0.01 for _, p in v)}/{len(v)} = {np.mean([p < 0.01 for _, p in v]):.3f} significant; "
          f"{len(SP) - len(v)} conditions not tested (fewer than 5 trials with at least 5 spikes on the track, or no variance)")
    vn = [r["prec"][k + "_null"] for r in SP if r["prec"] is not None and np.isfinite(r["prec"][k + "_null"][1])]
    print(f"   calibration, speed permuted over trials first: {np.mean([p < 0.01 for _, p in vn]):.4f} significant at p < 0.01 ({len(vn)} conditions)")
for k, nm in (("a", "per spike"), ("b", "100 ms bins")):
    v = [r["ratespeed"][k] for r in SP if r["ratespeed"] is not None and k in r["ratespeed"]]
    print(f"rate and speed {nm}: {sum(p < 0.01 for _, p in v)}/{len(v)} = {np.mean([p < 0.01 for _, p in v]):.3f} significant; median r {np.median([r for r, _ in v]):.3f}")

**Result: partially reproduced.** Both fractions are small, but both are below tolerance. Speed correlates with the rate of precession in 12 of 2,879 LS conditions (0.004, slope) or 26 of 2,877 (0.009, circular-linear r), against 0.028 +/- 0.0028; 1,483 conditions were not tested (fewer than 5 trials with at least 5 spikes on the track). With speed permuted first, the same test passes 0.0024 and 0.0049 of conditions, so it is conservative. Rate is modulated by speed in 0.058 (per spike) or 0.075 (100 ms bins) of 4,362 conditions, against 0.107 +/- 0.0107. Both are below 20%.

## fig5-assemblies
> Spike trains of hippocampal neurons predicted LS firing in 18.5% (2,087 of 11,264) of hippocampal-LS pairs, with a median optimal timescale of 25 ms.

In [ ]:
# Population of pairs: an LS unit and a hippocampal unit of the same session, both included in the condition
rng_pp = np.random.default_rng(5)
cand, all_cond_pairs = [], 0
for S in sessions:
    if not S["units"] or "trial_phase_ms" not in S:
        continue
    inc = {ui: set(tg.included_conditions(S, U)) for ui, U in enumerate(S["units"]) if "phase" in U}
    hp_ = [ui for ui in inc if S["units"][ui]["region"] in tg.HPC_LABELS]
    T = S["trials"]
    nh = sum(U["region"] in tg.HPC_LABELS for U in S["units"]); nl = sum(U["region"] == "ls" for U in S["units"])
    all_cond_pairs += nh * nl * sum((T["condition"] == c).sum() >= tg.MIN_TRIALS for c in np.unique(T["condition"]))
    for ul in inc:
        if S["units"][ul]["region"] != "ls":
            continue
        for c in inc[ul]:
            cand.append((S["session"], ul, c, sum(c in inc[uh] for uh in hp_)))
total_pairs = sum(x[3] for x in cand)
PP_TARGET = 6000
frac = min(1.0, PP_TARGET / total_pairs)
pick = [x for x in cand if rng_pp.random() < frac]
print(f"pairs by condition, all units (conditions with >= 10 trials): {all_cond_pairs}")
print(f"pairs by condition with both units included: {total_pairs}, over {len(cand)} LS unit-conditions; "
      f"sampled {len(pick)} LS unit-conditions ({frac:.3f}) with all their included hippocampal partners: {sum(x[3] for x in pick)} pairs")
picks = {}
for s_, ul, c, _ in pick:
    picks.setdefault(s_, []).append((ul, c))
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    pp_parts = list(ex.map(tg.peer_session, sessions, [picks.get(S["session"], []) for S in sessions], [seeds[S["session"]] + 4 for S in sessions]))
PP = pd.DataFrame([r for p in pp_parts for r in p])
print(f"{len(PP)} pairs fitted in {time.time()-t:.0f} s (150 windows x 11 fits each); strength not finite: {int(PP.strength.isna().sum())}")
PP["rule_b"] = (PP.best_idx > 7) & (PP.best_idx < 150) & (PP.mean_dev > 40) & (PP.ntrials > 10) & (PP.spikes_h + PP.spikes_l > 100)
PP["asm_a"] = PP.strength >= 4
PP["asm_b"] = PP.asm_a & PP.rule_b
for rd, col in (("a: Methods rule (strength >= 4)", "asm_a"), ("b: with makeFig5.m rules", "asm_b")):
    A = PP[PP[col]]
    print(f"reading {rd}: {len(A)}/{len(PP)} = {PP[col].mean():.3f} assemblies; median timescale w {A.best_w.median():.1f} ms, "
          f"full box 2w+1 {np.median(2 * A.best_w + 1):.1f} ms")
print("pairs passing the makeFig5.m rules (any strength):", int(PP.rule_b.sum()), "| estimated in the whole population:", round(PP.rule_b.mean() * total_pairs))
print("by animal, reading a:", PP.groupby("subject").asm_a.mean().round(3).to_dict(), "| pairs:", PP.subject.value_counts().to_dict())
print("strength quantiles (10, 50, 90%):", np.nanpercentile(PP.strength, [10, 50, 90]).round(2), "| best window at 1 ms:", int((PP.best_w == 1).sum()),
      "| at 150 ms:", int((PP.best_w == 150).sum()))
print("expected: fraction 0.185 +/- 0.0185, median timescale 25 +/- 2.5 ms, pairs examined 11,264 +/- 1,126.4")

**Result: not reproduced**, on a random sample and resting on assumptions about the fit. 348 of the 4,362 LS unit-conditions were drawn, with all 7,098 of their pairs with included hippocampal units. Under the Methods rule (a) 0.266 of pairs are assemblies, with a median best window of 4 ms (9 ms as the full box); with the makeFig5.m rules (b) 0.075, with 20 ms (41 ms). Neither fraction is within 0.185 +/- 0.0185 and no timescale reading is within 25 +/- 2.5 ms. The population holds 81,393 pairs with both units included (35,823 estimated to pass the makeFig5.m rules), against the paper's 11,264 pairs. 874 pairs had their best window at 1 ms and 269 at 150 ms, the ends of the range. The zero-lag strength rule of makeFig5.m is not applied. These assemblies match neither the paper's fraction nor its timescale, so the claims tested on them (fig5g, figs5, fig6e) are tested on a different assembly set from the paper's.

## fig5g-assembly-neurons-stronger-phase
> LS neurons that take part in at least one predictive assembly carry a stronger spatial phase code than those that do not (p < 1e-20).

In [ ]:
Dk = D.set_index(["session", "unit", "cond"])
PP["ls_d"] = [min(Dk.at[(s_, u, c), "aa1_mphase"], np.nanmean(Dk.at[(s_, u, c), "aa1_phase_raw"])) - Dk.at[(s_, u, c), "aa1_mrate"]
              for s_, u, c in zip(PP.session, PP.ls, PP.cond)]
for col in ("asm_a", "asm_b"):
    nc = PP.groupby(["session", "ls", "cond"]).agg(asm=(col, "any"), d=("ls_d", "first"))
    per_pair = PP[PP[col]].ls_d.values
    non = nc[~nc.asm].d.values
    for rd, a_ in (("(a) one value per assembly pair", per_pair), ("(b) per neuron and condition", nc[nc.asm].d.values)):
        t_ = stats.ttest_ind(a_, non); k_ = stats.ks_2samp(a_, non)
        print(f"{col}, {rd}: assembly {np.mean(a_):.0f} (n={len(a_)}) against non-assembly {np.mean(non):.0f} (n={len(non)}) "
              f"phase - rate MSE; t p = {t_.pvalue:.1e}, KS p = {k_.pvalue:.1e}")
print("phase MSE is the lower of the cosine and raw phase MSE (makeFig5.m), reading aa1")
print("variant, other decoder readings with the cosine phase only (the raw phase MSE is stored for aa1 alone):")
for rd in [r for r in RD if r[2] == "1"]:
    PP["_d"] = [Dk.at[(s_, u, c), f"{rd}_d"] for s_, u, c in zip(PP.session, PP.ls, PP.cond)]
    for col in ("asm_a", "asm_b"):
        nc = PP.groupby(["session", "ls", "cond"]).agg(asm=(col, "any"), d=("_d", "first"))
        a_, non = PP[PP[col]]._d.dropna().values, nc[~nc.asm].d.dropna().values
        b_ = nc[nc.asm].d.dropna().values
        print(f"   {rd}, {col}: per pair {np.mean(a_):.0f} (n={len(a_)}), per neuron {np.mean(b_):.0f} (n={len(b_)}), against {np.mean(non):.0f} (n={len(non)}); "
              f"t p = {stats.ttest_ind(a_, non).pvalue:.2g} and {stats.ttest_ind(b_, non).pvalue:.2g}, KS p = {stats.ks_2samp(a_, non).pvalue:.2g} and {stats.ks_2samp(b_, non).pvalue:.2g}")


**Result: not reproduced**, on the same sample. Under the Methods rule nearly every sampled LS neuron-condition is in an assembly (272 against 13), and the assembly values are less negative (-275 per pair, -293 per neuron, against -437; t test p = 0.52 and 0.58). With the makeFig5.m rules, -291 and -282 against -337 (p = 0.68 and 0.64). Assembly neurons do not have the more negative phase minus rate MSE. With the cosine phase only, under the other readings with phase 0, assembly neurons are less negative still (makeFig5.m rules: ba1 p = 0.017, bb1 p = 0.009, in the opposite direction). Only 13 sampled neuron-conditions are outside an assembly under the Methods rule, so that reading is a weak test.

## figs5-assembly-independent-of-locking
> Theta phase locking of hippocampal neurons does not significantly correlate with the strength or timescale of hippocampal-LS assemblies.

In [ ]:
lock = []
for s_, uh, c in zip(PP.session, PP.hpc, PP.cond):
    S = sess_by[s_]; U = S["units"][uh]
    m = np.isin(U["trial"], np.where(S["trials"]["condition"] == c)[0])
    R = float(np.abs(np.mean(np.exp(1j * U["phase"][m]))))
    lock.append((R, tg.circ_kappa(R)))
PP["lock_R"], PP["lock_kappa"] = zip(*lock)
for col in ("asm_a", "asm_b"):
    A = PP[PP[col] & np.isfinite(PP.strength)]
    out = []
    for lk in ("lock_kappa", "lock_R"):
        for y in ("strength", "best_w"):
            pr_, sp_ = stats.pearsonr(A[lk], A[y]), stats.spearmanr(A[lk], A[y])
            out.append(f"{lk} vs {y}: Pearson r = {pr_[0]:.3f} (p = {pr_[1]:.2g}), Spearman {sp_[0]:.3f} (p = {sp_[1]:.2g})")
    print(f"{col}, {len(A)} assembly pairs:"); print("   " + "\n   ".join(out))

**Result: reproduced**, on the same sample. A claim of absence, judged on Pearson correlation at p < 0.05. Over 1,890 assembly pairs (Methods rule) none of the four correlations of locking (kappa, resultant length) with strength or timescale is significant (r = 0.015 to 0.028, p = 0.23 to 0.52); over 534 pairs with the makeFig5.m rules none is either (r = -0.026 to 0.083, p = 0.056 to 0.62). The Spearman variant is significant for the 534 pairs (p = 0.019 and 0.036), and two Pearson p values are close to 0.05 (0.056 and 0.081), so the absence is fragile. The assembly set does not reproduce fig5-assemblies (0.266 or 0.075 against 0.185), so the absence is shown for a different set of assemblies from the paper's.

## fig6-session-counts
> LS neurons were recorded with CA1 neurons in 63 sessions and with CA3 neurons in 19 sessions, and 14 sessions had LFP from CA1, CA3 and LS together.

In [ ]:
g = UN.groupby("session").region.apply(set)
has_ls = g.apply(lambda s: "ls" in s)
hpc_s = g.apply(lambda s: bool(s & set(tg.HPC_LABELS)))
ca1_lab = g.apply(lambda s: bool(s & {"hpc", "ca1"})); ca3_lab = g.apply(lambda s: "ca3" in s)
ca3_site = g.index.isin(list(ca3_sessions))
print(f"LS with hippocampal units: {int((has_ls & hpc_s).sum())}")
print(f"session reading: LS with CA1 {int((has_ls & hpc_s & ~ca3_site).sum())}, LS with CA3 {int((has_ls & hpc_s & ca3_site).sum())}")
print(f"unit labels: LS with CA1 ('hpc'/'ca1') {int((has_ls & ca1_lab).sum())}, LS with CA3 ('ca3') {int((has_ls & ca3_lab).sum())}")
ls_ca1lfp = [S["session"] for S in sessions if S["units"] and any(U["region"] == "ls" for U in S["units"]) and S["chan"]["ca1"]]
ls_ca3lfp = [S["session"] for S in sessions if S["units"] and any(U["region"] == "ls" for U in S["units"]) and S["chan"]["ca3"]]
print(f"variant, by LFP channel: sessions with LS units and a CA1 channel {len(ls_ca1lfp)}, with a CA3 channel {len(ls_ca3lfp)}")
lfp3 = [S["session"] for S in sessions if S["chan"]["ca1"] and S["chan"]["ca3"] and S["chan"]["ls"]]
print(f"sessions with CA1, CA3 and LS LFP channels flagged: {len(lfp3)}; by animal {dict(Counter(sess_by[s]['subject'] for s in lfp3))}")
c1 = {"session reading": int((has_ls & hpc_s & ~ca3_site).sum()), "unit labels": int((has_ls & ca1_lab).sum())}
c3s = {"session reading": int((has_ls & hpc_s & ca3_site).sum()), "unit labels": int((has_ls & ca3_lab).sum())}
for k, v in c1.items():
    c3 = c3s[k]
    print(f"criterion, {k}: LS with CA1 {v} (63 +/- 6.3: {abs(v - 63) <= 6.3}), LS with CA3 {c3} (19 +/- 1.9: {abs(c3 - 19) <= 1.9}), "
          f"LFP {len(lfp3)} (14 +/- 1.4: {abs(len(lfp3) - 14) <= 1.4}); all three within: {abs(v - 63) <= 6.3 and abs(c3 - 19) <= 1.9 and abs(len(lfp3) - 14) <= 1.4}")
dt8 = [S["session"] for S in sessions if S["subject"] == "DT8" and S["units"] and any(U["region"] == "ls" for U in S["units"])]
print(f"DT8 sessions with LS units (no hippocampal units in the files): {len(dt8)}; LS with CA1 counted with all of them: "
      f"{c1['session reading'] + len(dt8)} or {c1['unit labels'] + len(dt8)}")


**Result: not reproduced** (agreed in the third verification round). The criterion has no partial rule: it is reproduced only when all three counts are within tolerance, which the line printed above shows they are not. LS units were recorded with CA3 in 19 sessions under the session reading (19 +/- 1.9), and 14 sessions have CA1, CA3 and LS channels flagged (14 +/- 1.4). LS and CA1 units are together in only 25 sessions (session reading) or 33 (unit labels), against 63. Counting a session's CA1 by its flagged CA1 channel gives 57, within tolerance, but that is a variant: the claim is about CA1 neurons. DT8 has hippocampal channels and a flagged CA1 channel in all 17 sessions but no hippocampal units (data check), which accounts for part of the shortfall: with all 17 DT8 LS sessions counted, LS with CA1 would be 42 or 50, still short of 63.

## fig6c-ca3-coherence-increases
> Theta phase coherence between LS and the CA3 pyramidal layer increases from the beginning to the end of the run, while coherence between LS and CA1 stays relatively flat.

In [ ]:
xs = np.linspace(0, 1, 1000)
def lfp_table(key):
    rows = []
    for S in sessions:
        L = S.get("lfp") or {}
        for reg in ("ca1", "ca3"):
            for r in L.get(reg, []):
                z = r[f"{key}_z"]
                if z is None:
                    continue
                rows.append({"session": S["session"], "subject": S["subject"], "reg": reg, "slope": np.polyfit(xs, z, 1)[0],
                             "raw": r[f"{key}_mean"], "z": z})
    return pd.DataFrame(rows)
def judge(T_, label):
    for lvl in ("trial", "session"):
        X = T_ if lvl == "trial" else T_.groupby(["session", "reg"]).agg(slope=("slope", "mean"), raw=("raw", "mean")).reset_index()
        a1, a3 = X[X.reg == "ca1"], X[X.reg == "ca3"]
        o1, o3 = stats.ttest_1samp(a1.slope, 0), stats.ttest_1samp(a3.slope, 0)
        d = stats.ttest_ind(a3.slope, a1.slope)
        r_ = stats.ttest_ind(a3.raw, a1.raw)
        print(f"{label}, per {lvl}: CA1 slope {a1.slope.mean():.3f} (n={len(a1)}, p={o1.pvalue:.1e}); CA3 slope {a3.slope.mean():.3f} "
              f"(n={len(a3)}, p={o3.pvalue:.1e}); CA3 - CA1 slope p={d.pvalue:.1e}; raw mean CA1 {a1.raw.mean():.4f}, CA3 {a3.raw.mean():.4f}, p={r_.pvalue:.1e}")
    for reg in ("ca1", "ca3"):
        Z = np.stack(T_[T_.reg == reg].z)
        m = Z.mean(0)
        print(f"   {reg} mean z curve: last tenth - first tenth = {m[-100:].mean() - m[:100].mean():.3f}; range {m.min():.2f} to {m.max():.2f}")
COH = lfp_table("coh")
print("sessions with a CA1 channel:", COH[COH.reg == "ca1"].session.nunique(), "| with a CA3 channel:", COH[COH.reg == "ca3"].session.nunique())
judge(COH, "theta phase coherence")
print("CA3 slope by animal:", COH[COH.reg == "ca3"].groupby("subject").slope.mean().round(3).to_dict(), "| CA1:", COH[COH.reg == "ca1"].groupby("subject").slope.mean().round(3).to_dict())

**Result: reproduced.** Per trial, the slope of z-scored CA3-LS theta phase coherence over the run is 0.366 (2,699 trials, p = 3.6e-111) and exceeds the CA1-LS slope of 0.044 (p = 4.6e-88). Over sessions the CA3 slope is 0.259 (19 sessions, p = 0.0062), above CA1 (p = 0.0030). The CA1 mean curve changes by 0.030 from first to last tenth against 0.321 for CA3. One of the three CA3 animals (DT7, mean slope -0.171) runs the other way.

## fig6d-cfc-stronger-ca3
> Coupling between hippocampal theta phase and LS gamma amplitude increases along the run for both CA1 and CA3, and is stronger for CA3-LS than for CA1-LS.

In [ ]:
CFC = lfp_table("cfc"); judge(CFC, "theta-gamma coupling, 40-200 Hz, 50 ms RMS")
CFC2 = lfp_table("cfc2"); judge(CFC2, "variant 50-500 Hz, 60-sample RMS")
same = CFC[CFC.session.isin(CFC[CFC.reg == "ca3"].session)]
judge(same, "40-200 Hz, only sessions with both channels")

**Result: reproduced.** Theta-gamma coupling rises over the run for CA1 (slope 0.084, p = 1.5e-19) and CA3 (0.284, p = 1.1e-49), and raw coupling is higher for CA3 (0.376 against 0.345, p = 5.3e-187); per session the same holds (p = 4.8e-05, 2.5e-06 and 7.0e-08). The 50 to 500 Hz variant gives the same signs.

## fig6e-assembly-position
> CA3 place fields form progressively stronger gamma assemblies with LS along the track (r = 0.34, p = 0.006) while CA1 coupling decreases somewhat (r = -0.11, p = 0.079), and the two are anti-correlated over position (r = -0.88).

In [ ]:
out = []
for pr in "ab":
    PP[f"com_{pr}"] = [Dk.at[(s_, u, c), f"pf{pr}_com"] if (s_, u, c) in Dk.index and Dk.at[(s_, u, c), f"pf{pr}_n"] > 0 else np.nan
                       for s_, u, c in zip(PP.session, PP.hpc, PP.cond)]
PP["sub_session"] = np.where(PP.session.isin(ca3_sessions), "CA3", "CA1")
PP["sub_label"] = np.where(PP.region == "ca3", "CA3", "CA1")
for pr in "ab":
    for which, sel in (("(a) assembly pairs", PP.asm_a), ("(b) all pairs with a field", PP.strength.notna())):
        for regk in ("sub_session", "sub_label"):
            X = PP[sel & PP[f"com_{pr}"].notna() & np.isfinite(PP.strength)]
            row = {"fields": pr, "pairs": which, "region": regk}
            curves = {}
            for reg in ("CA1", "CA3"):
                g = X[X[regk] == reg]
                r, p = stats.pearsonr(g[f"com_{pr}"], g.strength) if len(g) > 2 else (np.nan, np.nan)
                row[reg] = f"r={r:.3f} p={p:.2g} n={len(g)}"
                cv = np.full(201, np.nan)
                b = np.clip(np.round(g[f"com_{pr}"]).astype(int), 0, 200)
                for k, v in g.groupby(b).strength.mean().items():
                    cv[k] = v
                curves[reg] = cv
            for sm in ("none", "box20"):
                c1, c3 = curves["CA1"], curves["CA3"]
                if sm == "box20":
                    c1, c3 = tg.box_fast(c1[None], 20)[0], tg.box_fast(c3[None], 20)[0]
                ok = np.isfinite(c1) & np.isfinite(c3)
                row[f"curves_r_{sm}"] = f"{stats.pearsonr(c1[ok], c3[ok])[0]:.3f} (bins {ok.sum()})" if ok.sum() > 2 else "n/a"
            out.append(row)
pd.set_option("display.max_colwidth", 40)
print(pd.DataFrame(out).to_string(index=False))
print("expected: CA1 r -0.11 +/- 0.011 (not significant), CA3 r 0.34 +/- 0.034 (p < 0.05), curves r -0.88 +/- 0.088")

**Result: partially reproduced**, on the same sample. With Gaussian-map fields, assembly pairs and regions by session, the CA1 r is -0.106 (within -0.11 +/- 0.011, but significant, p = 0.024, 452 pairs), the CA3 r is +0.178 (p = 0.14, 71 pairs; 0.34 +/- 0.034) and the CA1 and CA3 strength curves correlate at -0.106 (unsmoothed, 42 bins with both) or -0.310 (20-bin box). The signs hold; the CA3 increase is not significant, the CA1 decrease is, and the curve correlation is far from -0.88. Other readings give CA3 r from -0.005 to 0.178, none significant; with fields (b) and the session reading the curve correlation turns positive. The assemblies are those of fig5-assemblies, which do not reproduce the paper's fraction or timescale.

## figs7-ensemble-ls-phase-best
> The firing phases of ensembles of simultaneously recorded LS neurons read out position better than the firing phases or rates of an equal number of hippocampal neurons (p < 0.001 at 15 to 25 neurons).

In [ ]:
ENS_DRAWS = 10   # reading (a): 10 random ensembles per size and condition
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    ens = [r for p in ex.map(tg.ensemble_session, sessions, [seeds[S["session"]] + 2 for S in sessions], [ENS_DRAWS] * len(sessions)) for r in p]
E = pd.DataFrame(ens)
print(f"{len(E)} ensemble decodes in {time.time()-t:.0f} s; conditions with >= 15 included units: "
      f"LS {E[E.region=='LS'].groupby(['session','cond']).ngroups}, HPC {E[E.region=='HPC'].groupby(['session','cond']).ngroups}")
for reading in ("a", "b"):
    rows = []
    for size in range(15, 26):
        e = E[(E['size'] == size) & (E.reading == reading)]
        lp = e[(e.region == "LS") & (e.feature == "phase")].mse; hp_ = e[(e.region == "HPC") & (e.feature == "phase")].mse; hr = e[(e.region == "HPC") & (e.feature == "rate")].mse
        p1 = stats.ttest_ind(lp, hp_).pvalue if len(lp) > 1 and len(hp_) > 1 else np.nan
        p2 = stats.ttest_ind(lp, hr).pvalue if len(lp) > 1 and len(hr) > 1 else np.nan
        rows.append({"size": size, "n_LS": len(lp), "LS_phase": round(lp.mean(), 0), "n_HPC": len(hp_), "HPC_phase": round(hp_.mean(), 0),
                     "HPC_rate": round(hr.mean(), 0), "p_vs_HPC_phase": p1, "p_vs_HPC_rate": p2,
                     "LS_best_p001": bool(lp.mean() < hp_.mean() and lp.mean() < hr.mean() and p1 < 0.001 and p2 < 0.001)})
    print(f"reading ({reading}):"); print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** At every size from 15 to 25, LS phase ensembles decode position worse than hippocampal phase ensembles (6,041 against 5,766 bins squared at 15 units, reading a, 10 draws) and much worse than hippocampal rate ensembles (2,005). The differences are significant in the wrong direction (p = 1.9e-26 and below 1e-300 at 15 units). Reading (b) gives the same.

## figs2-rate-phase-independent
> Instantaneous firing rate and firing phase are independent at the single-trial level; fewer than 2% of hippocampal conditions had a rate-phase correlation higher than a circular-shift null.

In [ ]:
t = time.time()
with ProcessPoolExecutor(4, mp_context=ctx) as ex:
    IF = pd.DataFrame([r for p in ex.map(tg.ifr_session, sessions, [seeds[S["session"]] + 3 for S in sessions]) for r in p])
print(f"{len(IF)} conditions in {time.time()-t:.0f} s")
for reg, m in (("hippocampal", IF.region.isin(tg.HPC_LABELS)), ("LS", IF.region == "ls")):
    X = IF[m]
    a_ = ((X.t_p < 0.05) & (X.t_stat > 0)).mean(); b_ = (X.mean_act > X.pct95).mean()
    print(f"{reg}: {len(X)} conditions; mean actual r {X.mean_act.mean():.3f}, mean shuffled {X.mean_sh.mean():.3f}; "
          f"(a) t test higher at p<0.05: {a_:.4f}; (b) above the 95th percentile: {b_:.4f}")

**Result: not reproduced.** Over 5,453 hippocampal conditions the mean actual rate-phase correlation (0.435) is no higher than the circularly shifted one (0.444). But 0.143 of conditions are significantly higher by the t test (a) and 0.207 lie above the 95th percentile (b), against the paper's fewer than 2%. For LS the fractions are 0.032 and 0.112.

## figs1f-trial-counts
> The behaviour comprises 2,889 circle-alternation, 6,608 central-alternation and 625 linear-track trials; ballistic trials made up about 90 to 95% of trials in each recording.

In [ ]:
print("trials by task, all sessions:", dict(tasks))
tasks_ls = Counter(x for S in sessions if S["units"] and any(U["region"] == "ls" for U in S["units"]) for x in task_of(S))
print("trials by task, sessions with LS units:", dict(tasks_ls))
desc = Counter("only ballistic runs" in S["trials_description"] for S in sessions)
print("trials tables whose description says only ballistic runs were kept:", desc.get(True, 0), "of", len(sessions))
print("example:", sessions[0]["trials_description"].split(". Start")[0])

**Result: not reproduced.** In the 62 sessions with LS units there are 3,039 circle (wheel) alternation trials (2,889 +/- 288.9) and 625 linear-track trials (625 +/- 62.5), but 5,219 central-alternation trials against 6,608 +/- 660.8. Over all 67 sessions the counts are 3,987, 5,219 and 701, all outside tolerance. The ballistic fraction cannot be tested: all 67 trials tables say that only ballistic runs were kept. The criterion has no partial rule, so two of three counts within tolerance is not a reproduction.

In [ ]:
print(f"notebook ran in {(time.time() - T0)/60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.1f} GB")

## Summary

| claim | outcome |
|---|---|
| ls-neuron-count | reproduced |
| hpc-unit-count | not reproduced |
| ls-active-neurons | reproduced |
| fig2-ls-phase-vs-shuffle | not reproduced (assumed) |
| fig2-ls-phase-beats-rate | not reproduced (assumed) |
| fig2e-ls-phase-lower-mse | not reproduced (assumed) |
| fig2fg-mse-difference | not reproduced (assumed) |
| fig1i-hpc-rate-phase-equivalent | not reproduced (assumed; reproduced proposed, not agreed) |
| ls-lower-peak-info | partially reproduced |
| ls-few-place-fields | partially reproduced |
| hpc-place-field-properties | partially reproduced |
| ls-place-fields-unstable | partially reproduced |
| fig1e-phase-info-smaller-scale | partially reproduced (assumed) |
| fig1g-phase-more-info-fine-scale | reproduced (assumed) |
| fig3-ls-phase-reliable | reproduced |
| fig4d-depth-gradient | reproduced (assumed) |
| fig4e-onset-depth | not reproduced (assumed) |
| fig4e-onset-mediolateral | partially reproduced (assumed) |
| speed-independent | partially reproduced |
| fig5-assemblies | not reproduced (assumed; sample of pairs) |
| fig5g-assembly-neurons-stronger-phase | not reproduced (assumed; sample of pairs) |
| figs5-assembly-independent-of-locking | reproduced (assumed; sample of pairs) |
| fig6-session-counts | not reproduced |
| fig6c-ca3-coherence-increases | reproduced |
| fig6d-cfc-stronger-ca3 | reproduced |
| fig6e-assembly-position | partially reproduced (assumed; sample of pairs) |
| figs7-ensemble-ls-phase-best | not reproduced (assumed) |
| figs2-rate-phase-independent | not reproduced |
| figs3-ls-major-target | blocked, other (Allen data not analysed in this run) |
| figs6-reference-frame | blocked, metadata_insufficient |
| figs1f-trial-counts | not reproduced |
| fig7-dynamic-weighting-model | blocked, other (model not run in this run) |

"proposed, not agreed" marks a change of outcome found in the final revision that the third verification round did not agree with; the outcome shown is the verified one.

"(assumed)" marks an outcome that rests on an assumption about a method the paper leaves open: above all, that position bins with no spike in their smoothing window take phase 0 in phase decoding and phase information, and that ties between equally near template bins are broken at random.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's.
- The paper does not say how bins without spikes enter phase decoding. They are set to phase 0 here. Leaving them out (the variant) lets the phase decoder read where the cell fires: synthetic place cells with random phases beat rate in all 200 cases under that variant, and real cells whose phases are shifted within trials beat rate about as often as the real data (decoder calibration). With phase 0, synthetic cells pass phase-against-rate in 0.005 or none, and shifted real phases in 0.026 to 0.060 of conditions, 3 to 6 times the nominal 1% but below the real 0.044 to 0.082; the LS phase claims fail and the hippocampal phase decoding is slightly worse than rate. At fine smoothing the phase-0 fill makes phase information measure whether a spike is near, which produces the fig1g match. The phase-against-chance test detects firing location under both treatments: shifted phases pass it nearly as often as real ones.
- Each criterion is applied as written (see the header). fig4d-depth-gradient keeps its round-2 outcome, reproduced on bb1 alone, although its criterion has no "under one reading" clause for the decoder readings; under aa1 it would likely be not reproduced.
- Ties between template bins (many bins of a rate template are exactly 0 Hz) are broken at random; sending them to the lowest bin, as a first-index rule does, is shown for the main reading.
- The split of samples (readings b) puts neighbouring, smoothed samples of one trial into both training and test sets; it is a reading of the claims file and is run, but the outcomes that rest on it alone (fig4d-depth-gradient) are fragile.
- Place fields follow the Methods' rules, applied the way buzcode's bz_getPlaceFields1D applies them. The Gaussian width (5 bins) and the choice of the highest peak as the "first" field are assumptions.
- Olypher information uses normalized probabilities; makeFig1.m's division by the trial count is shown as a variant. The best smoothing uses MATLAB's smooth over 5 points.
- Bins of 1.5 cm cannot be formed for DT2, whose positions are in pixels; those sessions keep 201 bins under that reading.
- Spike phase is referenced to the CA1 channel the files flag; six sessions without one use the hippocampal probe's theta channel (five DT9 sessions with only a CA3 probe, one DT7 session).
- Speed is computed from x and y position smoothed over 0.25 s; precession slope by maximizing the mean resultant length.
- Peer prediction (Figure 5) is run on a random 7% of the LS unit-conditions, with all their included hippocampal partners (7,098 pairs), by least squares, one fit per trial, cutting trials to the shorter in the shuffles.
- Which session-name depth is the LS probe's, and which way the sites run along a shank, are not stored; the first depth and sites deepening with their index are used, and the alternatives are shown.
- The Allen connectivity analysis (Figure S3) and the spiking model (Figure 7) were not run in this attempt's time.

What was found about the dataset.
- 31 sessions of DT7, DT8 and DT9 store head position labelled "meters" whose values are millimetres: their template paths are 1,329 to 2,934 units long, against 0.97 to 3.13 m elsewhere.
- One DT7 session (20170324) has no units table.
- No CA1 channel is flagged in five DT9 sessions (CA3 probe only) and one DT7 session. The paper refers all spike phases to CA1 theta.
- DT8 has hippocampal channels, with a flagged CA1 channel, in all 17 sessions but no hippocampal units.
- Hippocampal units number 3,260 (2,636 'hpc', 624 'ca3'), against the paper's 2,951. The 'hpc' label is the probe, not the subfield, so CA1 against CA3 is a reading.
- No session is marked as holding maze rotations, so Figure S6 cannot be tested.
- Some files lack trials.trial_type, trial_conditions.condition_type or units.max_waveform_channel.
- Clocks: 99.7% of linearized samples lie inside their trial, and hippocampal spikes lock to theta in 47 of 47 sessions (median resultant length 0.179 against 0.053 with spikes shifted by 1 s). Rate decoding with recorded positions beats positions 1 s later in 30 of 47 sessions with ties broken at random; the decoder is weak (median hippocampal rate MSE 6,329 against 6,733 for uniform guessing), so this check is consistent with aligned clocks but does not show it for every session. The sessions and the lowest-bin variant are listed in the data check.
